# Farol: nova inferência, adaptação do notebook original
As etapas analíticas originais foram preservadas. Ative os controles gradualmente; o padrão é piloto com inferências desligadas. O notebook original Windows está ao lado para rastreabilidade.

## Abrir o pacote no Colab
O código abaixo clona a branch de preparação no ambiente do Colab. Ajuste ROOT para uma cópia local já existente. Para inferência, selecione GPU no ambiente de execução; a reprodução dos CSVs usa CPU.

In [ ]:
from pathlib import Path
import sys, subprocess, os, json
ROOT = Path('/content/Estimacao_Distancia_Drone_TCC') if Path('/content').exists() else Path.cwd()
if not (ROOT/'README.md').is_file():
    subprocess.run(['git','clone','--branch','preparacao-tcc-2026-10-17','https://github.com/icarodeferreira/Estimacao_Distancia_Drone_TCC.git',str(ROOT)],check=True)
assert (ROOT/'scripts/reproduzir_farol.py').is_file(), 'Ajuste ROOT para a raiz do repositório.'
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-inference.txt'],check=True)

In [ ]:
subprocess.run([sys.executable,'scripts/instalar_depth_pro.py'],check=True)

# Pipeline temporal Farol — YOLO + Depth Pro + telemetria a 30 FPS

Experimento funcional temporal do TCC, criado especificamente para o vídeo recortado localizado em `C:\TCC_Drone_local\video_final_farol_original`.

Características do protocolo:

- caixa YOLO inferida em cada quadro;
- Depth Pro inferido em cada quadro;
- timestamp obtido em milissegundos do vídeo;
- telemetria de aproximadamente 10 Hz interpolada para cada quadro;
- sincronização por evento físico de decolagem;
- distância nominal e intervalo geométrico em relação à câmera;
- execução retomável por CSV;
- piloto obrigatório antes do processamento integral;
- vídeo anotado e métricas temporais.

> Este é um experimento temporal complementar. Quadros consecutivos não são amostras estatisticamente independentes; por isso, as métricas agregadas por segundo são tratadas como resumo principal.


## 1. Ambiente, caminhos e controles

Execute esta seção primeiro. Na primeira passagem, mantenha `RUN_MODE = "pilot"` e todos os controles de processamento como `False`.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

ROOT = Path.cwd()
INPUT_DIR = ROOT / "inputs" / "video"
DEPTH_PRO_REPO = ROOT / "external" / "ml-depth-pro"
DEPTH_CHECKPOINT = DEPTH_PRO_REPO / "checkpoints" / "depth_pro.pt"

DETECTOR_TAG = "yolo_proprio"
YOLO_WEIGHTS = (
    ROOT / "treinos_transferencia"
    / "yolo26n_aod4_finetune_own_1920_seed42"
    / "weights" / "best.pt"
)

# Se houver mais de um vídeo ou CSV na pasta, preencha explicitamente.
VIDEO_PATH_OVERRIDE: Optional[str] = (
    r"C:\TCC_Drone_local\video_final_farol_original"
    r"\video_final_farol.mov"
)
TELEMETRY_PATH_OVERRIDE: Optional[str] = None

YOLO_WEIGHTS=ROOT/'models/best.pt'
VIDEO_PATH_OVERRIDE=str(ROOT/'inputs/video/farol.mov')
TELEMETRY_PATH_OVERRIDE=str(ROOT/'data/telemetry/farol_original.csv')

# Sincronização por evento físico e timecode CapCut HH:MM:SS:FF.
TIMECODE_FPS = 30.0
ORIGINAL_VIDEO_TAKEOFF_TIMECODE = "00:00:00:10"
CROP_START_TIMECODE = "00:00:02:25"
ORIGINAL_VIDEO_TAKEOFF_S = 10 / TIMECODE_FPS
TELEMETRY_TAKEOFF_S = 2.3
CROP_START_IN_ORIGINAL_VIDEO_S = 2 + 25 / TIMECODE_FPS
TELEMETRY_AT_CROP_START_S = (
    TELEMETRY_TAKEOFF_S
    + CROP_START_IN_ORIGINAL_VIDEO_S
    - ORIGINAL_VIDEO_TAKEOFF_S
)  # 4,8 s

# Geometria informada.
CAMERA_HEIGHT_M = 1.5
CAMERA_TO_HOME_HORIZONTAL_M = 1.5

# Configuração Depth Pro congelada no experimento anterior.
FIXED_FOCAL_PX = 1440.0
CAL_INTERCEPT = 1.4600571394959514
CAL_SLOPE = 0.5026751660343942

# Primeiro execute pilot; somente depois altere para full.
RUN_MODE = "pilot"              # "pilot" ou "full"
PILOT_SECONDS = 10.0
YOLO_IMGSZ = 1920
YOLO_CONF_FLOOR = 0.001
YOLO_RELIABLE_CONF = 0.25
MIN_BOX_WIDTH_PX = 10.0
MIN_BOX_HEIGHT_PX = 5.0
MIN_VALID_DEPTH_PIXELS = 6
MIN_ESTIMATE_M = 1.0
MAX_ESTIMATE_M = 150.0
ACCEPT_ABS_M = 5.0
ACCEPT_REL = 0.20
SAVE_EVERY_YOLO = 25
SAVE_EVERY_DEPTH = 5
ROI_DIAGNOSTIC_FRAMES_PER_ZONE = 20

RUN_VIDEO_AUDIT = False
RUN_YOLO_ALL_FRAMES = False
RUN_DEPTH_ALL_FRAMES = False
RUN_TEMPORAL_METRICS = False
RUN_ROI_DIAGNOSTIC = False
RUN_ANNOTATED_VIDEO = False

REUSE_LEGACY_INFERENCE = False
FORCE_REFRESH_SYNCHRONIZATION = False
FORCE_RESTART = False

if RUN_MODE not in {"pilot", "full"}:
    raise ValueError("RUN_MODE deve ser 'pilot' ou 'full'.")

# A sincronização corrigida usa pasta própria para não misturar resultados antigos.
SYNC_TAG = "sync_tc_00000225_telemetry_004800"
LEGACY_OUTPUT_DIR = (
    ROOT / "resultados_pipeline_farol_30fps"
    / DETECTOR_TAG / RUN_MODE
)
OUTPUT_DIR = LEGACY_OUTPUT_DIR / SYNC_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEGACY_YOLO_CSV = LEGACY_OUTPUT_DIR / "yolo_all_frames.csv"
LEGACY_DEPTH_CSV = LEGACY_OUTPUT_DIR / "depth_all_frames.csv"
YOLO_CSV = OUTPUT_DIR / "yolo_all_frames.csv"
DEPTH_CSV = OUTPUT_DIR / "depth_all_frames.csv"
PER_SECOND_CSV = OUTPUT_DIR / "metrics_per_second.csv"
ZONE_CSV = OUTPUT_DIR / "metrics_by_zone.csv"
METRICS_JSON = OUTPUT_DIR / "temporal_metrics.json"
ANNOTATED_VIDEO = OUTPUT_DIR / "pipeline_farol_30fps_annotated.mp4"  # legado Depth Pro
FINAL_ANNOTATED_VIDEO = OUTPUT_DIR / "pipeline_farol_box_width_final_annotated.mp4"
FINAL_REPORT_MD = OUTPUT_DIR / "relatorio_final_metodos.md"
FINAL_SUMMARY_JSON = OUTPUT_DIR / "resumo_final_metodos.json"
PROTOCOL_JSON = OUTPUT_DIR / "temporal_protocol.json"

for p, label in [
    (INPUT_DIR, "pasta de entrada"),
    (DEPTH_CHECKPOINT, "checkpoint Depth Pro"),
    (YOLO_WEIGHTS, "pesos YOLO próprio"),
]:
    if not p.exists():
        raise FileNotFoundError(f"{label} não encontrado: {p}")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Modo:", RUN_MODE)
print("Timecode da decolagem:", ORIGINAL_VIDEO_TAKEOFF_TIMECODE)
print("Timecode do início do recorte:", CROP_START_TIMECODE)
print("Início do recorte no original:", CROP_START_IN_ORIGINAL_VIDEO_S, "s")
print("Início correspondente na telemetria:", TELEMETRY_AT_CROP_START_S, "s")
print("Saída:", OUTPUT_DIR)


In [ ]:
# Controle do diagnóstico de recortes (seção 8.4).
RUN_CROP_DEPTH_DIAGNOSTIC = False
# Calibração exploratória dos recortes; não executa nova inferência.
RUN_CROP_CALIBRATION_CV = False
# Validação por segundo em toda a sequência; não executa inferência.
RUN_FULL_BOX_CALIBRATION_CV = False
# Auditoria do modelo de largura; não executa inferência.
RUN_BOX_ROBUSTNESS_AUDIT = False

RUN_FINAL_REPORT = False
RUN_FINAL_ANNOTATED_VIDEO = True

BOX_AUDIT_MIN_FRAMES_PER_SECOND = 5
BOX_AUDIT_MAX_RELATIVE_IQR = 0.35
BOX_AUDIT_HAMPEL_HISTORY_SECONDS = 5
BOX_AUDIT_HAMPEL_K = 3.0
BOX_AUDIT_MAX_LOG_WIDTH_JUMP = 0.35
CROP_DIAGNOSTIC_SIDES_PX = (512, 1024)


## 2. Descoberta dos arquivos, leitura da telemetria e auditoria da sincronização

O CSV do DJI contém uma primeira linha `sep=,`; esta seção a trata automaticamente. O timecode CapCut `00:00:02:25` corresponde ao quadro 85 do original e o primeiro quadro do recorte é associado a `4,8 s` da telemetria.


In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".mts"}

def discover_unique(override: Optional[str], extensions: set[str], label: str) -> Path:
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    found = sorted(
        p for p in INPUT_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )
    if len(found) != 1:
        print(f"{label}s encontrados:")
        for path in found:
            print(" -", path)
        raise RuntimeError(f"Defina {label.upper()}_PATH_OVERRIDE explicitamente.")
    return found[0]

VIDEO_PATH = discover_unique(VIDEO_PATH_OVERRIDE, VIDEO_EXTENSIONS, "video")
TELEMETRY_PATH = discover_unique(TELEMETRY_PATH_OVERRIDE, {".csv"}, "telemetry")

with TELEMETRY_PATH.open("r", encoding="utf-8-sig", errors="replace") as handle:
    first_line = handle.readline().strip()
skiprows = 1 if first_line.lower().startswith("sep=") else 0
telemetry = pd.read_csv(TELEMETRY_PATH, skiprows=skiprows, low_memory=False)

required_columns = [
    "OSD.flyTime [s]", "OSD.height [ft]", "HOME.distance [ft]",
    "OSD.latitude", "OSD.longitude",
]
missing = [column for column in required_columns if column not in telemetry.columns]
if missing:
    raise KeyError(f"Colunas ausentes na telemetria: {missing}")

for column in required_columns:
    telemetry[column] = pd.to_numeric(telemetry[column], errors="coerce")

telemetry = (
    telemetry.dropna(subset=["OSD.flyTime [s]"])
    .sort_values("OSD.flyTime [s]")
    .drop_duplicates("OSD.flyTime [s]", keep="last")
    .reset_index(drop=True)
)
telemetry["height_m"] = telemetry["OSD.height [ft]"] * 0.3048
telemetry["home_distance_m"] = telemetry["HOME.distance [ft]"] * 0.3048

telemetry_t = telemetry["OSD.flyTime [s]"].to_numpy(float)
telemetry_height_m = telemetry["height_m"].to_numpy(float)
telemetry_home_m = telemetry["home_distance_m"].to_numpy(float)
telemetry_lat = telemetry["OSD.latitude"].to_numpy(float)
telemetry_lon = telemetry["OSD.longitude"].to_numpy(float)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Não foi possível abrir {VIDEO_PATH}")
NATIVE_FPS = float(cap.get(cv2.CAP_PROP_FPS))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION_S = TOTAL_FRAMES / NATIVE_FPS
cap.release()

FULL_END_FRAME = min(
    TOTAL_FRAMES,
    max(0, int(math.floor((telemetry_t.max() - TELEMETRY_AT_CROP_START_S) * NATIVE_FPS)) + 1),
)
END_FRAME_EXCLUSIVE = (
    min(FULL_END_FRAME, int(round(PILOT_SECONDS * NATIVE_FPS)))
    if RUN_MODE == "pilot" else FULL_END_FRAME
)
START_FRAME = 0
TARGET_FRAMES = END_FRAME_EXCLUSIVE

telemetry_sampling_hz = 1.0 / np.median(np.diff(telemetry_t))
sync_summary = {
    "video": str(VIDEO_PATH),
    "telemetry": str(TELEMETRY_PATH),
    "native_fps": NATIVE_FPS,
    "total_video_frames": TOTAL_FRAMES,
    "target_frames": TARGET_FRAMES,
    "video_duration_s": DURATION_S,
    "resolution": [WIDTH, HEIGHT],
    "telemetry_range_s": [float(telemetry_t.min()), float(telemetry_t.max())],
    "telemetry_sampling_hz_approx": float(telemetry_sampling_hz),
    "original_video_takeoff_s": ORIGINAL_VIDEO_TAKEOFF_S,
    "telemetry_takeoff_s": TELEMETRY_TAKEOFF_S,
    "crop_start_in_original_video_s": CROP_START_IN_ORIGINAL_VIDEO_S,
    "telemetry_at_crop_start_s": TELEMETRY_AT_CROP_START_S,
}
print(json.dumps(sync_summary, indent=2, ensure_ascii=False))


## 3. Funções de interpolação, geometria, ROI e abstinência

A distância exata câmera–drone depende do ângulo entre a câmera e o deslocamento horizontal do drone. Como esse ângulo não foi medido, o notebook registra:

- `reference_distance_nominal_m`: aproximação com componentes ortogonais;
- `reference_distance_min_partial_m` e `reference_distance_max_partial_m`: limites decorrentes do deslocamento horizontal de 1,5 m da câmera;
- `interval_error_partial_m`: erro zero quando a estimativa cai dentro desse intervalo.


In [ ]:
def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def interp_finite(t: float, x: np.ndarray, y: np.ndarray) -> float:
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 2 or t < x[valid].min() or t > x[valid].max():
        return np.nan
    return float(np.interp(t, x[valid], y[valid]))

def telemetry_record_at(telemetry_time_s: float) -> dict:
    height_m = interp_finite(telemetry_time_s, telemetry_t, telemetry_height_m)
    home_m = interp_finite(telemetry_time_s, telemetry_t, telemetry_home_m)
    latitude = interp_finite(telemetry_time_s, telemetry_t, telemetry_lat)
    longitude = interp_finite(telemetry_time_s, telemetry_t, telemetry_lon)
    if not np.isfinite(height_m) or not np.isfinite(home_m):
        return {
            "drone_height_m": height_m, "home_distance_m": home_m,
            "latitude": latitude, "longitude": longitude,
            "reference_distance_nominal_m": np.nan,
            "reference_distance_min_partial_m": np.nan,
            "reference_distance_max_partial_m": np.nan,
        }
    dz = height_m - CAMERA_HEIGHT_M
    horizontal_min = abs(home_m - CAMERA_TO_HOME_HORIZONTAL_M)
    horizontal_max = home_m + CAMERA_TO_HOME_HORIZONTAL_M
    nominal = math.sqrt(home_m**2 + CAMERA_TO_HOME_HORIZONTAL_M**2 + dz**2)
    lower = math.sqrt(horizontal_min**2 + dz**2)
    upper = math.sqrt(horizontal_max**2 + dz**2)
    return {
        "drone_height_m": height_m, "home_distance_m": home_m,
        "latitude": latitude, "longitude": longitude,
        "reference_distance_nominal_m": nominal,
        "reference_distance_min_partial_m": lower,
        "reference_distance_max_partial_m": upper,
    }

def frame_timestamp_s(cap: cv2.VideoCapture, frame_idx: int) -> tuple[float, str]:
    nominal = frame_idx / NATIVE_FPS
    pos_msec = float(cap.get(cv2.CAP_PROP_POS_MSEC))
    measured = pos_msec / 1000.0
    if np.isfinite(measured) and measured >= 0 and abs(measured - nominal) <= 0.25:
        return measured, "pos_msec"
    return nominal, "frame_index_fallback"

def distance_zone(distance_m: float) -> str:
    if not np.isfinite(distance_m):
        return "outside_telemetry"
    if distance_m < 20:
        return "critical_0_20m"
    if distance_m < 40:
        return "attention_20_40m"
    if distance_m < 80:
        return "distant_40_80m"
    return "very_distant_80m_plus"

def calibrate(raw_m: float) -> float:
    if not np.isfinite(raw_m) or raw_m <= 0:
        return np.nan
    return float(np.exp(CAL_INTERCEPT + CAL_SLOPE * np.log(raw_m)))

def acceptable(estimate: float, reference: float) -> bool:
    return bool(
        np.isfinite(estimate) and np.isfinite(reference)
        and abs(estimate-reference) <= max(ACCEPT_ABS_M, ACCEPT_REL*reference)
    )

def partial_interval_error(estimate: float, lower: float, upper: float) -> float:
    if not all(np.isfinite([estimate, lower, upper])):
        return np.nan
    if lower <= estimate <= upper:
        return 0.0
    return float(min(abs(estimate-lower), abs(estimate-upper)))

def parse_box(row) -> Optional[list[float]]:
    values = [row.get(k, np.nan) for k in ["x1", "y1", "x2", "y2"]]
    return list(map(float, values)) if np.all(np.isfinite(values)) else None

def roi_p25(depth: np.ndarray, box, source_wh) -> tuple[float, int]:
    if box is None:
        return np.nan, 0
    sw, sh = source_wh
    dh, dw = depth.shape
    x1, y1, x2, y2 = map(float, box)
    ix1 = int(np.floor(np.clip(x1*dw/sw, 0, dw-1)))
    ix2 = int(np.ceil(np.clip(x2*dw/sw, ix1+1, dw)))
    iy1 = int(np.floor(np.clip(y1*dh/sh, 0, dh-1)))
    iy2 = int(np.ceil(np.clip(y2*dh/sh, iy1+1, dh)))
    values = depth[iy1:iy2, ix1:ix2].reshape(-1)
    values = values[np.isfinite(values) & (values > 0)]
    return (float(np.percentile(values, 25)), int(values.size)) if values.size else (np.nan, 0)

def abstention_reason(conf, box, valid_pixels, estimate) -> str:
    if box is None:
        return "no_detection"
    x1, y1, x2, y2 = box
    if not np.isfinite(conf) or conf < YOLO_RELIABLE_CONF:
        return "low_confidence"
    if x2-x1 < MIN_BOX_WIDTH_PX or y2-y1 < MIN_BOX_HEIGHT_PX:
        return "box_too_small"
    if valid_pixels < MIN_VALID_DEPTH_PIXELS:
        return "insufficient_depth_pixels"
    if not np.isfinite(estimate) or not MIN_ESTIMATE_M <= estimate <= MAX_ESTIMATE_M:
        return "invalid_distance"
    return ""


## 4. Verificação inicial e estimativa de duração

Confirme especialmente:

- FPS próximo de 29,97/30;
- `telemetry_at_crop_start_s = 4.8`;
- referência inicial calculada com a telemetria corrigida, sem expectativa baseada no alinhamento antigo;
- duração e quantidade de quadros do piloto.


In [ ]:
if RUN_VIDEO_AUDIT:
    first_reference = telemetry_record_at(TELEMETRY_AT_CROP_START_S)
    print("Referência do primeiro quadro:")
    print(json.dumps(first_reference, indent=2, ensure_ascii=False))
    print(f"Quadros a processar: {TARGET_FRAMES}")
    print(f"Duração representada: {TARGET_FRAMES/NATIVE_FPS:.3f} s")
    print("Estimativa baseada no histórico de 1,4–2,5 s por inferência Depth Pro:")
    print(f"Depth Pro: {TARGET_FRAMES*1.4/3600:.2f}–{TARGET_FRAMES*2.5/3600:.2f} horas")
    if RUN_MODE == "full":
        print("ATENÇÃO: modo integral selecionado.")


## 4.1. Reaproveitamento seguro das inferências com sincronização corrigida

Esta etapa preserva os resultados anteriores e cria cópias na pasta da sincronização corrigida. Somente as colunas temporais, a telemetria interpolada, as distâncias de referência e as zonas são recalculadas. Bounding boxes, confiança, tempos de inferência e estimativas Depth Pro permanecem inalterados.

In [ ]:
def relabel_inference_csv(source_path: Path, target_path: Path, label: str) -> None:
    if target_path.exists() and not FORCE_REFRESH_SYNCHRONIZATION:
        relabel_source = target_path
        print(f"Atualizando referências e métricas derivadas de {label}: {target_path}")
    elif source_path.exists():
        relabel_source = source_path
    else:
        print(f"{label} anterior não encontrado; será gerado normalmente quando liberado: {source_path}")
        return

    frame_results = pd.read_csv(relabel_source, low_memory=False)
    if "video_time_s" not in frame_results.columns:
        raise KeyError(f"video_time_s ausente em {source_path}")

    frame_results["original_video_time_s"] = (
        frame_results["video_time_s"].astype(float)
        + CROP_START_IN_ORIGINAL_VIDEO_S
    )
    frame_results["telemetry_time_s"] = (
        frame_results["video_time_s"].astype(float)
        + TELEMETRY_AT_CROP_START_S
    )

    corrected_references = pd.DataFrame([
        telemetry_record_at(float(t))
        for t in frame_results["telemetry_time_s"]
    ], index=frame_results.index)

    for column in corrected_references.columns:
        frame_results[column] = corrected_references[column]

    frame_results["distance_zone"] = frame_results[
        "reference_distance_nominal_m"
    ].apply(distance_zone)

    # Estimativas Depth Pro independem da referência, mas todas as métricas
    # derivadas precisam ser recalculadas depois de qualquer nova sincronização.
    if "estimated_distance_m" in frame_results.columns:
        estimate_values = pd.to_numeric(
            frame_results["estimated_distance_m"], errors="coerce"
        ).to_numpy(float)
        reference_values = pd.to_numeric(
            frame_results["reference_distance_nominal_m"], errors="coerce"
        ).to_numpy(float)
        lower_values = pd.to_numeric(
            frame_results["reference_distance_min_partial_m"], errors="coerce"
        ).to_numpy(float)
        upper_values = pd.to_numeric(
            frame_results["reference_distance_max_partial_m"], errors="coerce"
        ).to_numpy(float)
        abstained_values = frame_results.get(
            "abstained", pd.Series(False, index=frame_results.index)
        ).fillna(True).astype(bool).to_numpy()

        frame_results["signed_error_m"] = estimate_values-reference_values
        frame_results["absolute_error_m"] = np.abs(estimate_values-reference_values)
        frame_results["interval_error_partial_m"] = [
            partial_interval_error(estimate, lower, upper)
            for estimate, lower, upper in zip(
                estimate_values, lower_values, upper_values
            )
        ]
        frame_results["acceptable"] = [
            acceptable(estimate, reference) if not abstained else False
            for estimate, reference, abstained in zip(
                estimate_values, reference_values, abstained_values
            )
        ]

    atomic_csv(frame_results.sort_values("frame_idx"), target_path)
    print(
        f"{label} reaproveitado com sincronização corrigida: "
        f"{len(frame_results)} registros -> {target_path}"
    )


if REUSE_LEGACY_INFERENCE:
    relabel_inference_csv(LEGACY_YOLO_CSV, YOLO_CSV, "YOLO")
    relabel_inference_csv(LEGACY_DEPTH_CSV, DEPTH_CSV, "Depth Pro")

sync_correction_record = {
    "timecode_format": "HH:MM:SS:FF",
    "timecode_fps": TIMECODE_FPS,
    "takeoff_timecode": ORIGINAL_VIDEO_TAKEOFF_TIMECODE,
    "takeoff_original_s": ORIGINAL_VIDEO_TAKEOFF_S,
    "crop_start_timecode": CROP_START_TIMECODE,
    "crop_start_original_s": CROP_START_IN_ORIGINAL_VIDEO_S,
    "telemetry_takeoff_s": TELEMETRY_TAKEOFF_S,
    "telemetry_at_crop_start_s": TELEMETRY_AT_CROP_START_S,
    "legacy_output_dir": str(LEGACY_OUTPUT_DIR),
    "corrected_output_dir": str(OUTPUT_DIR),
    "reused_inference": REUSE_LEGACY_INFERENCE,
}
with (OUTPUT_DIR / "synchronization_correction.json").open("w", encoding="utf-8") as handle:
    json.dump(sync_correction_record, handle, indent=2, ensure_ascii=False)
print(json.dumps(sync_correction_record, indent=2, ensure_ascii=False))

## 5. Passagem 1 — YOLO em todos os quadros

Para o piloto, altere apenas `RUN_YOLO_ALL_FRAMES = True` na seção 1, reexecute as seções 1–4 e então esta seção. Depois volte o controle para `False`.


In [ ]:
def choose_drone_ids(model) -> set[int]:
    names = model.names if isinstance(model.names, dict) else dict(enumerate(model.names))
    ids = {int(k) for k, v in names.items() if "drone" in str(v).lower() or "uav" in str(v).lower()}
    if not ids and len(names) == 1:
        ids = {int(next(iter(names)))}
    if not ids:
        raise ValueError(f"Classe drone não encontrada: {names}")
    print("Classes:", {i: names[i] for i in ids})
    return ids

def run_yolo_all_frames() -> pd.DataFrame:
    from ultralytics import YOLO
    done = pd.DataFrame() if FORCE_RESTART or not YOLO_CSV.exists() else pd.read_csv(YOLO_CSV)
    completed = set(done.get("frame_idx", pd.Series(dtype=int)).astype(int))
    records = done.to_dict("records")
    model = YOLO(str(YOLO_WEIGHTS))
    class_ids = choose_drone_ids(model)
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    for frame_idx in range(END_FRAME_EXCLUSIVE):
        ok, frame = cap.read()
        if not ok:
            break
        video_time_s, timestamp_source = frame_timestamp_s(cap, frame_idx)
        if frame_idx in completed:
            continue
        started = time.perf_counter()
        result = model.predict(
            frame, imgsz=YOLO_IMGSZ, conf=YOLO_CONF_FLOOR,
            verbose=False, device=0 if torch.cuda.is_available() else "cpu",
        )[0]
        elapsed = time.perf_counter() - started
        candidates = []
        if result.boxes is not None:
            for box, conf, cls in zip(
                result.boxes.xyxy.detach().cpu().numpy(),
                result.boxes.conf.detach().cpu().numpy(),
                result.boxes.cls.detach().cpu().numpy().astype(int),
            ):
                if int(cls) in class_ids:
                    candidates.append((box.astype(float).tolist(), float(conf), int(cls)))
        chosen = max(candidates, key=lambda item: item[1], default=None)
        box = chosen[0] if chosen else [np.nan]*4
        telemetry_time_s = video_time_s + TELEMETRY_AT_CROP_START_S
        reference = telemetry_record_at(telemetry_time_s)
        nominal = reference["reference_distance_nominal_m"]
        records.append({
            "frame_idx": frame_idx,
            "video_time_ms": video_time_s*1000.0,
            "video_time_s": video_time_s,
            "timestamp_source": timestamp_source,
            "original_video_time_s": video_time_s + CROP_START_IN_ORIGINAL_VIDEO_S,
            "telemetry_time_s": telemetry_time_s,
            **reference,
            "distance_zone": distance_zone(nominal),
            "detection_count": len(candidates),
            "confidence": chosen[1] if chosen else np.nan,
            "class_id": chosen[2] if chosen else np.nan,
            "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3],
            "yolo_time_s": elapsed, "yolo_imgsz": YOLO_IMGSZ,
        })
        if len(records) % SAVE_EVERY_YOLO == 0:
            atomic_csv(pd.DataFrame(records).sort_values("frame_idx"), YOLO_CSV)
            print(f"YOLO: {len(records)}/{TARGET_FRAMES}")
    cap.release()
    result_df = pd.DataFrame(records).drop_duplicates("frame_idx", keep="last").sort_values("frame_idx")
    atomic_csv(result_df, YOLO_CSV)
    print(f"YOLO concluído: {len(result_df)}/{TARGET_FRAMES}")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df

if RUN_YOLO_ALL_FRAMES:
    yolo_frames = run_yolo_all_frames()
elif YOLO_CSV.exists():
    yolo_frames = pd.read_csv(YOLO_CSV)
    print(f"YOLO já disponível: {len(yolo_frames)}/{TARGET_FRAMES}")
else:
    yolo_frames = pd.DataFrame()
    print("YOLO 30 FPS bloqueado.")


## 6. Diagnóstico rápido do YOLO

Execute depois da seção 5. A baixa confiança é esperada em alvos muito distantes, mas deve ser quantificada antes de gastar horas com o Depth Pro.


In [ ]:
if len(yolo_frames):
    diagnostic = {
        "frames": int(len(yolo_frames)),
        "frames_with_detection": int(yolo_frames["confidence"].notna().sum()),
        "detection_coverage_pct": float(yolo_frames["confidence"].notna().mean()*100),
        "reliable_confidence_frames": int((yolo_frames["confidence"] >= YOLO_RELIABLE_CONF).sum()),
        "reliable_confidence_pct": float((yolo_frames["confidence"] >= YOLO_RELIABLE_CONF).mean()*100),
        "confidence_median": float(yolo_frames["confidence"].median()),
        "box_width_median_px": float((yolo_frames["x2"]-yolo_frames["x1"]).median()),
        "box_height_median_px": float((yolo_frames["y2"]-yolo_frames["y1"]).median()),
    }
    print(json.dumps(diagnostic, indent=2, ensure_ascii=False, allow_nan=True))
    display(yolo_frames.head())
else:
    print("Execute a passagem YOLO.")


## 7. Passagem 2 — Depth Pro em todos os quadros

Após concluir e conferir o YOLO, mantenha `RUN_YOLO_ALL_FRAMES = False`, altere `RUN_DEPTH_ALL_FRAMES = True`, reexecute as seções 1–4 e execute esta seção.


In [ ]:
def load_depth_model():
    if str(DEPTH_PRO_REPO / "src") not in sys.path:
        sys.path.insert(0, str(DEPTH_PRO_REPO / "src"))
    import depth_pro
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    precision = torch.float16 if device.type == "cuda" else torch.float32
    cwd = Path.cwd()
    try:
        os.chdir(DEPTH_PRO_REPO)
        model, transform = depth_pro.create_model_and_transforms(device=device, precision=precision)
    finally:
        os.chdir(cwd)
    return model.eval(), transform, device, precision

def run_depth_all_frames() -> pd.DataFrame:
    if not YOLO_CSV.exists():
        raise FileNotFoundError("Execute a passagem YOLO primeiro.")
    yolo = pd.read_csv(YOLO_CSV).set_index("frame_idx")
    if len(yolo) != TARGET_FRAMES:
        raise RuntimeError(f"YOLO incompleto: {len(yolo)}/{TARGET_FRAMES}")
    done = pd.DataFrame() if FORCE_RESTART or not DEPTH_CSV.exists() else pd.read_csv(DEPTH_CSV)
    completed = set(done.get("frame_idx", pd.Series(dtype=int)).astype(int))
    records = done.to_dict("records")
    model, transform, device, precision = load_depth_model()
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    for frame_idx in range(END_FRAME_EXCLUSIVE):
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx in completed:
            continue
        row = yolo.loc[frame_idx]
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).convert("RGB")
        tensor = transform(pil).to(device=device, dtype=precision)
        focal = torch.tensor([FIXED_FOCAL_PX], device=device, dtype=tensor.dtype)
        if device.type == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            prediction = model.infer(tensor, f_px=focal)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        depth = prediction["depth"].detach().float().cpu().numpy().squeeze()
        box = parse_box(row)
        raw, pixels = roi_p25(depth, box, pil.size)
        estimate = calibrate(raw)
        reason = abstention_reason(float(row.get("confidence", np.nan)), box, pixels, estimate)
        reference = float(row.reference_distance_nominal_m)
        lower = float(row.reference_distance_min_partial_m)
        upper = float(row.reference_distance_max_partial_m)
        rec = row.to_dict()
        rec.update({
            "frame_idx": frame_idx,
            "raw_s00_p25_m": raw,
            "valid_depth_pixels": pixels,
            "estimated_distance_m": estimate,
            "abstained": bool(reason),
            "abstention_reason": reason,
            "signed_error_m": estimate-reference if np.isfinite(reference) and np.isfinite(estimate) else np.nan,
            "absolute_error_m": abs(estimate-reference) if np.isfinite(reference) and np.isfinite(estimate) else np.nan,
            "interval_error_partial_m": partial_interval_error(estimate, lower, upper),
            "acceptable": acceptable(estimate, reference) if not reason else False,
            "depth_time_s": elapsed,
        })
        records.append(rec)
        if len(records) % SAVE_EVERY_DEPTH == 0:
            atomic_csv(pd.DataFrame(records).sort_values("frame_idx"), DEPTH_CSV)
            print(f"Depth: {len(records)}/{TARGET_FRAMES}")
    cap.release()
    result_df = pd.DataFrame(records).drop_duplicates("frame_idx", keep="last").sort_values("frame_idx")
    atomic_csv(result_df, DEPTH_CSV)
    print(f"Depth concluído: {len(result_df)}/{TARGET_FRAMES}")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result_df

if RUN_DEPTH_ALL_FRAMES:
    depth_frames = run_depth_all_frames()
elif DEPTH_CSV.exists():
    depth_frames = pd.read_csv(DEPTH_CSV)
    print(f"Depth já disponível: {len(depth_frames)}/{TARGET_FRAMES}")
else:
    depth_frames = pd.DataFrame()
    print("Depth 30 FPS bloqueado.")


## 8. Métricas temporais

Depois de concluir o Depth Pro, altere apenas `RUN_TEMPORAL_METRICS = True`, reexecute as seções 1–4 e execute esta seção.


In [ ]:
def metric_dict(reference, estimate) -> dict:
    ref = np.asarray(reference, float)
    est = np.asarray(estimate, float)
    valid = np.isfinite(ref) & np.isfinite(est)
    ref, est = ref[valid], est[valid]
    if not len(ref):
        return {"n": 0}
    err = est-ref
    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((ref-np.mean(ref))**2))
    r2 = 1.0 - ss_res/ss_tot if len(ref) > 1 and ss_tot > 0 else np.nan
    return {
        "n": int(len(ref)),
        "mae_m": float(np.mean(np.abs(err))),
        "rmse_m": float(np.sqrt(np.mean(err**2))),
        "median_ae_m": float(np.median(np.abs(err))),
        "mape_pct": float(np.mean(np.abs(err)/np.maximum(np.abs(ref), 1e-9))*100),
        "bias_m": float(np.mean(err)),
        "r2": float(r2),
        "spearman": float(pd.Series(ref).corr(pd.Series(est), method="spearman")) if len(ref) > 1 else np.nan,
    }

def longest_true_run(values) -> int:
    best = current = 0
    for value in values:
        current = current + 1 if bool(value) else 0
        best = max(best, current)
    return best

if RUN_TEMPORAL_METRICS:
    if not len(depth_frames):
        raise RuntimeError("Execute o Depth Pro antes das métricas.")
    df = depth_frames.copy()

    # Recálculo defensivo: nunca confiar em erros/aceitabilidade persistidos
    # por uma sincronização anterior.
    estimate_values = pd.to_numeric(df["estimated_distance_m"], errors="coerce").to_numpy(float)
    reference_values = pd.to_numeric(df["reference_distance_nominal_m"], errors="coerce").to_numpy(float)
    lower_values = pd.to_numeric(df["reference_distance_min_partial_m"], errors="coerce").to_numpy(float)
    upper_values = pd.to_numeric(df["reference_distance_max_partial_m"], errors="coerce").to_numpy(float)
    abstained_values = df["abstained"].fillna(True).astype(bool).to_numpy()
    df["signed_error_m"] = estimate_values-reference_values
    df["absolute_error_m"] = np.abs(estimate_values-reference_values)
    df["interval_error_partial_m"] = [
        partial_interval_error(estimate, lower, upper)
        for estimate, lower, upper in zip(estimate_values, lower_values, upper_values)
    ]
    df["acceptable"] = [
        acceptable(estimate, reference) if not abstained else False
        for estimate, reference, abstained in zip(
            estimate_values, reference_values, abstained_values
        )
    ]

    in_scope = df["reference_distance_nominal_m"].notna()
    emitted = in_scope & ~df["abstained"].astype(bool)
    valid = df.loc[emitted].copy()
    valid["second"] = np.floor(valid["video_time_s"]).astype(int)
    per_second = valid.groupby("second", as_index=False).agg(
        video_time_s=("video_time_s", "median"),
        telemetry_time_s=("telemetry_time_s", "median"),
        reference_distance_nominal_m=("reference_distance_nominal_m", "median"),
        reference_distance_min_partial_m=("reference_distance_min_partial_m", "median"),
        reference_distance_max_partial_m=("reference_distance_max_partial_m", "median"),
        estimated_distance_m=("estimated_distance_m", "median"),
        frames=("frame_idx", "size"),
        confidence=("confidence", "median"),
    )
    atomic_csv(per_second, PER_SECOND_CSV)

    zone_rows = []
    for zone, part in df.loc[in_scope].groupby("distance_zone"):
        part_emitted = part.loc[~part["abstained"].astype(bool)]
        zone_rows.append({
            "distance_zone": zone,
            "frames": int(len(part)),
            "emission_coverage_pct": float(len(part_emitted)/max(1, len(part))*100),
            "acceptable_pct_all_frames": float(part["acceptable"].astype(bool).mean()*100),
            "mean_interval_error_partial_m": float(part_emitted["interval_error_partial_m"].mean()),
            **metric_dict(part_emitted.reference_distance_nominal_m, part_emitted.estimated_distance_m),
        })
    zones = pd.DataFrame(zone_rows)
    atomic_csv(zones, ZONE_CSV)

    estimate_series = valid["estimated_distance_m"].to_numpy(float)
    longest_abstention = longest_true_run(df.loc[in_scope, "abstained"])
    metrics = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "mode": RUN_MODE,
        "frames_processed": int(len(df)),
        "telemetry_labeled_frames": int(in_scope.sum()),
        "emitted_frames": int(emitted.sum()),
        "emission_coverage_pct": float(emitted.sum()/max(1, in_scope.sum())*100),
        "acceptable_pct_all_labeled_frames": float((df["acceptable"].astype(bool) & in_scope).sum()/max(1, in_scope.sum())*100),
        "per_frame_descriptive": metric_dict(valid.reference_distance_nominal_m, valid.estimated_distance_m),
        "per_second_primary_temporal": metric_dict(per_second.reference_distance_nominal_m, per_second.estimated_distance_m),
        "mean_partial_interval_error_m": float(valid["interval_error_partial_m"].mean()),
        "median_frame_to_frame_jitter_m": float(np.median(np.abs(np.diff(estimate_series)))) if len(estimate_series)>1 else np.nan,
        "longest_abstention_run_frames": longest_abstention,
        "longest_abstention_run_s": longest_abstention/NATIVE_FPS,
        "mean_yolo_time_s": float(df["yolo_time_s"].mean()),
        "mean_depth_time_s": float(df["depth_time_s"].mean()),
        "offline_real_time_factor": float((df["yolo_time_s"].sum()+df["depth_time_s"].sum()) / (len(df)/NATIVE_FPS)),
        "abstention_reasons": df.loc[df["abstained"].astype(bool), "abstention_reason"].value_counts().to_dict(),
        "scientific_note": "Quadros adjacentes e rótulos interpolados são temporalmente dependentes; o resumo por segundo é primário.",
    }
    with METRICS_JSON.open("w", encoding="utf-8") as handle:
        json.dump(metrics, handle, indent=2, ensure_ascii=False, allow_nan=True)
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    display(zones.round(3))
    display(per_second.head())
else:
    print("Métricas bloqueadas.")


## 8.1. Comparação completa — Depth bruto versus calibração antiga

Esta seção usa somente `depth_all_frames.csv`; não executa novamente YOLO nem Depth Pro. Ela separa os critérios estritos (`erro ≤ 1 m` e `erro relativo ≤ 20%`) do critério legado (`5 m ou 20%`), calcula resultados por quadro, por segundo e por zona, e salva um painel diagnóstico.


In [ ]:
if RUN_TEMPORAL_METRICS:
    import matplotlib.pyplot as plt

    required = [
        "frame_idx", "video_time_s", "reference_distance_nominal_m",
        "raw_s00_p25_m", "estimated_distance_m", "distance_zone",
        "abstained",
    ]
    missing = [column for column in required if column not in depth_frames.columns]
    if missing:
        raise KeyError(f"Colunas ausentes no Depth CSV: {missing}")

    comparison = depth_frames.copy()
    for column in [
        "reference_distance_nominal_m", "raw_s00_p25_m",
        "estimated_distance_m", "video_time_s",
    ]:
        comparison[column] = pd.to_numeric(comparison[column], errors="coerce")

    labeled = comparison["reference_distance_nominal_m"].notna()
    emitted = labeled & ~comparison["abstained"].fillna(True).astype(bool)
    finite_methods = (
        np.isfinite(comparison["raw_s00_p25_m"])
        & np.isfinite(comparison["estimated_distance_m"])
    )
    valid_comparison = comparison.loc[emitted & finite_methods].copy()
    valid_comparison["second"] = np.floor(valid_comparison["video_time_s"]).astype(int)

    method_columns = {
        "Depth bruto — focal 1440 px": "raw_s00_p25_m",
        "Depth calibrado — calibração antiga": "estimated_distance_m",
    }

    def acceptance_dict(reference, estimate, all_labeled_count=None):
        reference = np.asarray(reference, float)
        estimate = np.asarray(estimate, float)
        finite = np.isfinite(reference) & np.isfinite(estimate)
        reference, estimate = reference[finite], estimate[finite]
        absolute = np.abs(estimate-reference)
        relative = absolute/np.maximum(np.abs(reference), 1e-9)
        within_1m = absolute <= 1.0
        within_20pct = relative <= 0.20
        legacy = absolute <= np.maximum(5.0, 0.20*np.abs(reference))
        result = {
            "within_1m_pct_emitted": float(within_1m.mean()*100) if len(absolute) else np.nan,
            "within_20pct_pct_emitted": float(within_20pct.mean()*100) if len(absolute) else np.nan,
            "legacy_5m_or_20pct_pct_emitted": float(legacy.mean()*100) if len(absolute) else np.nan,
        }
        if all_labeled_count is not None:
            denominator = max(1, int(all_labeled_count))
            result.update({
                "within_1m_pct_all_labeled": float(within_1m.sum()/denominator*100),
                "within_20pct_pct_all_labeled": float(within_20pct.sum()/denominator*100),
                "legacy_5m_or_20pct_pct_all_labeled": float(legacy.sum()/denominator*100),
            })
        return result

    comparison_per_second = valid_comparison.groupby("second", as_index=False).agg(
        video_time_s=("video_time_s", "median"),
        reference_distance_nominal_m=("reference_distance_nominal_m", "median"),
        raw_s00_p25_m=("raw_s00_p25_m", "median"),
        estimated_distance_m=("estimated_distance_m", "median"),
        frames=("frame_idx", "size"),
        confidence=("confidence", "median"),
    )
    comparison_per_second["raw_absolute_error_m"] = np.abs(
        comparison_per_second["raw_s00_p25_m"]
        - comparison_per_second["reference_distance_nominal_m"]
    )
    comparison_per_second["calibrated_absolute_error_m"] = np.abs(
        comparison_per_second["estimated_distance_m"]
        - comparison_per_second["reference_distance_nominal_m"]
    )

    summary_rows = []
    for method, estimate_column in method_columns.items():
        frame_reference = valid_comparison["reference_distance_nominal_m"]
        frame_estimate = valid_comparison[estimate_column]
        summary_rows.append({
            "level": "per_frame_descriptive",
            "method": method,
            "emission_coverage_pct": float(len(valid_comparison)/max(1, labeled.sum())*100),
            "mean_estimate_m": float(frame_estimate.mean()),
            "median_estimate_m": float(frame_estimate.median()),
            **metric_dict(frame_reference, frame_estimate),
            **acceptance_dict(frame_reference, frame_estimate, labeled.sum()),
        })
        second_reference = comparison_per_second["reference_distance_nominal_m"]
        second_estimate = comparison_per_second[estimate_column]
        summary_rows.append({
            "level": "per_second_primary",
            "method": method,
            "emission_coverage_pct": np.nan,
            "mean_estimate_m": float(second_estimate.mean()),
            "median_estimate_m": float(second_estimate.median()),
            **metric_dict(second_reference, second_estimate),
            **acceptance_dict(second_reference, second_estimate),
        })
    method_summary = pd.DataFrame(summary_rows)

    zone_rows = []
    for zone, zone_all in comparison.loc[labeled].groupby("distance_zone"):
        zone_valid = valid_comparison.loc[valid_comparison["distance_zone"] == zone]
        for method, estimate_column in method_columns.items():
            reference = zone_valid["reference_distance_nominal_m"]
            estimate = zone_valid[estimate_column]
            zone_rows.append({
                "distance_zone": zone,
                "method": method,
                "frames_labeled": int(len(zone_all)),
                "frames_emitted": int(len(zone_valid)),
                "emission_coverage_pct": float(len(zone_valid)/max(1, len(zone_all))*100),
                "mean_estimate_m": float(estimate.mean()),
                "median_estimate_m": float(estimate.median()),
                **metric_dict(reference, estimate),
                **acceptance_dict(reference, estimate, len(zone_all)),
            })
    method_by_zone = pd.DataFrame(zone_rows)

    METHOD_SUMMARY_CSV = OUTPUT_DIR / "method_comparison_summary.csv"
    METHOD_PER_SECOND_CSV = OUTPUT_DIR / "method_comparison_per_second.csv"
    METHOD_BY_ZONE_CSV = OUTPUT_DIR / "method_comparison_by_zone.csv"
    METHOD_PLOT = OUTPUT_DIR / "method_comparison_diagnostic.png"
    atomic_csv(method_summary, METHOD_SUMMARY_CSV)
    atomic_csv(comparison_per_second, METHOD_PER_SECOND_CSV)
    atomic_csv(method_by_zone, METHOD_BY_ZONE_CSV)

    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].plot(
        comparison_per_second["video_time_s"],
        comparison_per_second["reference_distance_nominal_m"],
        color="black", linewidth=2.2, label="Referência",
    )
    axes[0].plot(
        comparison_per_second["video_time_s"],
        comparison_per_second["raw_s00_p25_m"],
        alpha=.85, label="Depth bruto",
    )
    axes[0].plot(
        comparison_per_second["video_time_s"],
        comparison_per_second["estimated_distance_m"],
        alpha=.85, label="Depth calibrado",
    )
    axes[0].set(title="Distância por segundo", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[0].grid(alpha=.25)
    axes[0].legend()

    scatter_reference = valid_comparison["reference_distance_nominal_m"]
    axes[1].scatter(scatter_reference, valid_comparison["raw_s00_p25_m"], s=8, alpha=.22, label="Bruto")
    axes[1].scatter(scatter_reference, valid_comparison["estimated_distance_m"], s=8, alpha=.22, label="Calibrado")
    diagonal_max = float(np.nanmax([scatter_reference.max(), valid_comparison["raw_s00_p25_m"].max(), valid_comparison["estimated_distance_m"].max()]))
    axes[1].plot([0, diagonal_max], [0, diagonal_max], "k--", linewidth=1.5, label="Ideal")
    axes[1].set(title="Estimativa versus referência", xlabel="Referência (m)", ylabel="Estimativa (m)")
    axes[1].grid(alpha=.25)
    axes[1].legend()

    zone_order = [zone for zone in ["critical_0_20m", "attention_20_40m", "distant_40_80m"] if zone in set(method_by_zone["distance_zone"])]
    x = np.arange(len(zone_order))
    width = .36
    for index, (method, _) in enumerate(method_columns.items()):
        values = (
            method_by_zone.loc[method_by_zone["method"] == method]
            .set_index("distance_zone").reindex(zone_order)["mae_m"]
        )
        axes[2].bar(x + (index-.5)*width, values, width, label=method)
    axes[2].set_xticks(x, zone_order)
    axes[2].set(title="MAE por faixa de distância", xlabel="Faixa", ylabel="MAE (m)")
    axes[2].grid(axis="y", alpha=.25)
    axes[2].legend()

    fig.savefig(METHOD_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print("Resumo bruto versus calibrado:")
    display(method_summary.round(3))
    print("Comparação por zona:")
    display(method_by_zone.round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Comparação de métodos bloqueada junto com as métricas.")


## 8.2. Sensibilidade à focal e limite de uma correção global

Execute depois da seção 8.1. O Depth Pro não será executado novamente. Como a profundidade métrica calculada com focal conhecida varia linearmente com `f_px`, esta seção reescala a saída já salva, testa a focal automática mediana observada (`2985,761 px`) e calcula focais globais otimizadas **somente como diagnóstico interno**. As focais otimizadas usam a própria referência deste vídeo e não constituem resultado de validação independente.


In [ ]:
if RUN_TEMPORAL_METRICS:
    if "valid_comparison" not in globals() or "comparison_per_second" not in globals():
        raise RuntimeError("Execute primeiro as seções 8 e 8.1.")

    AUTO_FOCAL_MEDIAN_PX = 2985.761
    focal_grid = np.linspace(800.0, 12000.0, 225)
    second_reference = comparison_per_second["reference_distance_nominal_m"].to_numpy(float)
    second_raw = comparison_per_second["raw_s00_p25_m"].to_numpy(float)

    focal_rows = []
    for focal_px in focal_grid:
        estimate = second_raw * (focal_px/FIXED_FOCAL_PX)
        scores = metric_dict(second_reference, estimate)
        focal_rows.append({
            "focal_px": float(focal_px),
            **scores,
            **acceptance_dict(second_reference, estimate),
        })
    focal_sensitivity = pd.DataFrame(focal_rows)
    best_mae_row = focal_sensitivity.loc[focal_sensitivity["mae_m"].idxmin()]
    best_rmse_row = focal_sensitivity.loc[focal_sensitivity["rmse_m"].idxmin()]
    BEST_MAE_FOCAL_DIAGNOSTIC_PX = float(best_mae_row["focal_px"])
    BEST_RMSE_FOCAL_DIAGNOSTIC_PX = float(best_rmse_row["focal_px"])

    focal_candidates = {
        "Focal fixa original — 1440 px": FIXED_FOCAL_PX,
        "Focal automática mediana — 2985,761 px": AUTO_FOCAL_MEDIAN_PX,
        "Melhor focal por MAE — diagnóstico interno": BEST_MAE_FOCAL_DIAGNOSTIC_PX,
        "Melhor focal por RMSE — diagnóstico interno": BEST_RMSE_FOCAL_DIAGNOSTIC_PX,
    }
    focal_candidate_rows = []
    for method, focal_px in focal_candidates.items():
        estimate = second_raw * (focal_px/FIXED_FOCAL_PX)
        focal_candidate_rows.append({
            "method": method,
            "focal_px": float(focal_px),
            "mean_estimate_m": float(np.mean(estimate)),
            "median_estimate_m": float(np.median(estimate)),
            **metric_dict(second_reference, estimate),
            **acceptance_dict(second_reference, estimate),
        })
    old_calibrated_second = comparison_per_second["estimated_distance_m"].to_numpy(float)
    focal_candidate_rows.append({
        "method": "Calibração log-linear antiga",
        "focal_px": np.nan,
        "mean_estimate_m": float(np.mean(old_calibrated_second)),
        "median_estimate_m": float(np.median(old_calibrated_second)),
        **metric_dict(second_reference, old_calibrated_second),
        **acceptance_dict(second_reference, old_calibrated_second),
    })
    focal_candidate_summary = pd.DataFrame(focal_candidate_rows)

    required_focal = valid_comparison.loc[
        np.isfinite(valid_comparison["raw_s00_p25_m"])
        & (valid_comparison["raw_s00_p25_m"] > 0)
    ].copy()
    required_focal["required_focal_px"] = (
        FIXED_FOCAL_PX
        * required_focal["reference_distance_nominal_m"]
        / required_focal["raw_s00_p25_m"]
    )
    required_focal_by_zone = required_focal.groupby("distance_zone", as_index=False).agg(
        n=("required_focal_px", "size"),
        reference_mean_m=("reference_distance_nominal_m", "mean"),
        required_focal_p25_px=("required_focal_px", lambda values: float(np.percentile(values, 25))),
        required_focal_median_px=("required_focal_px", "median"),
        required_focal_p75_px=("required_focal_px", lambda values: float(np.percentile(values, 75))),
    )
    required_focal_spearman = float(
        required_focal["reference_distance_nominal_m"].corr(
            required_focal["required_focal_px"], method="spearman"
        )
    )

    FOCAL_SENSITIVITY_CSV = OUTPUT_DIR / "focal_sensitivity_per_second.csv"
    FOCAL_CANDIDATES_CSV = OUTPUT_DIR / "focal_candidate_summary.csv"
    REQUIRED_FOCAL_BY_ZONE_CSV = OUTPUT_DIR / "required_focal_by_zone.csv"
    FOCAL_DIAGNOSTIC_PLOT = OUTPUT_DIR / "focal_sensitivity_diagnostic.png"
    atomic_csv(focal_sensitivity, FOCAL_SENSITIVITY_CSV)
    atomic_csv(focal_candidate_summary, FOCAL_CANDIDATES_CSV)
    atomic_csv(required_focal_by_zone, REQUIRED_FOCAL_BY_ZONE_CSV)

    fig, axes = plt.subplots(3, 1, figsize=(14, 17), constrained_layout=True)
    axes[0].plot(focal_sensitivity["focal_px"], focal_sensitivity["mae_m"], label="MAE")
    axes[0].plot(focal_sensitivity["focal_px"], focal_sensitivity["rmse_m"], label="RMSE")
    axes[0].axvline(FIXED_FOCAL_PX, color="tab:blue", linestyle="--", label="Fixa 1440 px")
    axes[0].axvline(AUTO_FOCAL_MEDIAN_PX, color="tab:green", linestyle="--", label="Auto mediana 2985,761 px")
    axes[0].axvline(BEST_MAE_FOCAL_DIAGNOSTIC_PX, color="tab:red", linestyle=":", label="Mínimo MAE interno")
    axes[0].set(title="Sensibilidade do erro à focal — resumo por segundo", xlabel="Focal (px)", ylabel="Erro (m)")
    axes[0].grid(alpha=.25)
    axes[0].legend()

    axes[1].plot(comparison_per_second["video_time_s"], second_reference, color="black", linewidth=2.2, label="Referência")
    axes[1].plot(comparison_per_second["video_time_s"], second_raw, alpha=.75, label="Fixa 1440 px")
    axes[1].plot(comparison_per_second["video_time_s"], second_raw*(AUTO_FOCAL_MEDIAN_PX/FIXED_FOCAL_PX), alpha=.8, label="Auto mediana")
    axes[1].plot(comparison_per_second["video_time_s"], second_raw*(BEST_MAE_FOCAL_DIAGNOSTIC_PX/FIXED_FOCAL_PX), alpha=.8, label="Melhor MAE interno")
    axes[1].plot(comparison_per_second["video_time_s"], old_calibrated_second, alpha=.75, label="Calibração antiga")
    axes[1].set(title="Efeito da focal na série temporal", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[1].grid(alpha=.25)
    axes[1].legend(ncol=2)

    axes[2].scatter(
        required_focal["reference_distance_nominal_m"],
        required_focal["required_focal_px"],
        s=8, alpha=.20,
    )
    axes[2].axhline(AUTO_FOCAL_MEDIAN_PX, color="tab:green", linestyle="--", label="Auto mediana")
    axes[2].set(title=f"Focal exigida por quadro (Spearman com distância = {required_focal_spearman:.3f})", xlabel="Referência (m)", ylabel="Focal necessária (px)")
    axes[2].grid(alpha=.25)
    axes[2].legend()

    fig.savefig(FOCAL_DIAGNOSTIC_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print("Comparação das focais candidatas — valores otimizados são apenas diagnósticos internos:")
    display(focal_candidate_summary.round(3))
    print("Focal necessária por faixa:")
    display(required_focal_by_zone.round(3))
    print(f"Spearman entre referência e focal necessária: {required_focal_spearman:.3f}")
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Diagnóstico de focal bloqueado junto com as métricas.")


## 8.3. Diagnóstico amostral da ROI do drone

Ative `RUN_ROI_DIAGNOSTIC = True` somente após as seções 8–8.2. Esta etapa seleciona até 20 quadros distribuídos no tempo em cada faixa de distância (máximo de 60), executa novamente o Depth Pro apenas nessa amostra e compara percentis em três regiões: caixa completa, região interna de 60% e núcleo central de 30%. O objetivo é verificar se a estatística `p25` da caixa completa está sendo dominada pelo fundo. Tempo esperado com o desempenho observado: aproximadamente 4–6 minutos. O CSV parcial permite retomada.


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
def roi_percentile_set(depth: np.ndarray, box, source_wh, shrink_fraction: float) -> dict:
    if box is None:
        return {f"p{q:02d}": np.nan for q in [5, 10, 25, 50]} | {"pixels": 0}
    sw, sh = source_wh
    dh, dw = depth.shape
    x1, y1, x2, y2 = map(float, box)
    box_w, box_h = x2-x1, y2-y1
    x1 += shrink_fraction*box_w
    x2 -= shrink_fraction*box_w
    y1 += shrink_fraction*box_h
    y2 -= shrink_fraction*box_h
    ix1 = int(np.floor(np.clip(x1*dw/sw, 0, dw-1)))
    ix2 = int(np.ceil(np.clip(x2*dw/sw, ix1+1, dw)))
    iy1 = int(np.floor(np.clip(y1*dh/sh, 0, dh-1)))
    iy2 = int(np.ceil(np.clip(y2*dh/sh, iy1+1, dh)))
    values = depth[iy1:iy2, ix1:ix2].reshape(-1)
    values = values[np.isfinite(values) & (values > 0)]
    result = {"pixels": int(values.size)}
    for q in [5, 10, 25, 50]:
        result[f"p{q:02d}"] = float(np.percentile(values, q)) if values.size else np.nan
    return result


if RUN_ROI_DIAGNOSTIC:
    if RUN_MODE != "full":
        raise RuntimeError("Use RUN_MODE='full' para o diagnóstico estratificado.")
    if not len(depth_frames):
        raise RuntimeError("Carregue depth_all_frames.csv antes do diagnóstico.")

    eligible = depth_frames.loc[
        depth_frames["reference_distance_nominal_m"].notna()
        & ~depth_frames["abstained"].fillna(True).astype(bool)
    ].sort_values("frame_idx")
    selected_parts = []
    for zone, part in eligible.groupby("distance_zone"):
        count = min(ROI_DIAGNOSTIC_FRAMES_PER_ZONE, len(part))
        positions = np.unique(np.rint(np.linspace(0, len(part)-1, count)).astype(int))
        selected_parts.append(part.iloc[positions])
    selected = (
        pd.concat(selected_parts, ignore_index=True)
        .drop_duplicates("frame_idx")
        .sort_values("frame_idx")
    )

    ROI_DIAGNOSTIC_CSV = OUTPUT_DIR / "roi_depth_diagnostic_frames.csv"
    ROI_DIAGNOSTIC_SUMMARY_CSV = OUTPUT_DIR / "roi_depth_diagnostic_summary.csv"
    ROI_DIAGNOSTIC_PLOT = OUTPUT_DIR / "roi_depth_diagnostic.png"
    if ROI_DIAGNOSTIC_CSV.exists() and not FORCE_RESTART:
        roi_records_df = pd.read_csv(ROI_DIAGNOSTIC_CSV)
        roi_records_df = roi_records_df.loc[
            roi_records_df["frame_idx"].isin(selected["frame_idx"])
        ].drop_duplicates("frame_idx", keep="last")
    else:
        roi_records_df = pd.DataFrame()
    completed = set(roi_records_df.get("frame_idx", pd.Series(dtype=int)).astype(int))
    missing_rows = selected.loc[~selected["frame_idx"].astype(int).isin(completed)]

    if len(missing_rows):
        model, transform, device, precision = load_depth_model()
        cap = cv2.VideoCapture(str(VIDEO_PATH))
        new_records = []
        for progress, (_, row) in enumerate(missing_rows.iterrows(), start=1):
            frame_idx = int(row["frame_idx"])
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame = cap.read()
            if not ok:
                print(f"Falha ao ler quadro {frame_idx}")
                continue
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).convert("RGB")
            tensor = transform(pil).to(device=device, dtype=precision)
            focal = torch.tensor([FIXED_FOCAL_PX], device=device, dtype=tensor.dtype)
            if device.type == "cuda":
                torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                prediction = model.infer(tensor, f_px=focal)
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter()-started
            depth = prediction["depth"].detach().float().cpu().numpy().squeeze()
            box = parse_box(row)
            record = {
                "frame_idx": frame_idx,
                "video_time_s": float(row["video_time_s"]),
                "reference_distance_nominal_m": float(row["reference_distance_nominal_m"]),
                "distance_zone": row["distance_zone"],
                "confidence": float(row["confidence"]),
                "box_width_px": float(row["x2"]-row["x1"]),
                "box_height_px": float(row["y2"]-row["y1"]),
                "depth_time_s": elapsed,
            }
            for prefix, shrink in [("full", 0.00), ("inner60", 0.20), ("center30", 0.35)]:
                stats = roi_percentile_set(depth, box, pil.size, shrink)
                record[f"{prefix}_pixels"] = stats["pixels"]
                for q in [5, 10, 25, 50]:
                    record[f"{prefix}_p{q:02d}_m"] = stats[f"p{q:02d}"]
            new_records.append(record)
            combined = pd.concat([roi_records_df, pd.DataFrame(new_records)], ignore_index=True)
            atomic_csv(combined.drop_duplicates("frame_idx", keep="last").sort_values("frame_idx"), ROI_DIAGNOSTIC_CSV)
            print(f"ROI diagnóstico: {len(completed)+progress}/{len(selected)}")
        cap.release()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    roi_diagnostic = pd.read_csv(ROI_DIAGNOSTIC_CSV).sort_values("frame_idx")
    raw_columns = [
        f"{region}_p{q:02d}_m"
        for region in ["full", "inner60", "center30"]
        for q in [5, 10, 25, 50]
    ]
    roi_summary_rows = []
    for raw_column in raw_columns:
        for focal_label, focal_px in [
            ("fixed_1440", FIXED_FOCAL_PX),
            ("auto_median_2985_761", AUTO_FOCAL_MEDIAN_PX),
        ]:
            reference = roi_diagnostic["reference_distance_nominal_m"]
            estimate = roi_diagnostic[raw_column]*(focal_px/FIXED_FOCAL_PX)
            roi_summary_rows.append({
                "roi_statistic": raw_column,
                "focal_mode": focal_label,
                "focal_px": focal_px,
                "mean_estimate_m": float(estimate.mean()),
                "median_estimate_m": float(estimate.median()),
                **metric_dict(reference, estimate),
                **acceptance_dict(reference, estimate),
            })
    roi_summary = pd.DataFrame(roi_summary_rows)
    atomic_csv(roi_summary, ROI_DIAGNOSTIC_SUMMARY_CSV)

    fixed_summary = roi_summary.loc[roi_summary["focal_mode"] == "fixed_1440"].copy()
    auto_summary = roi_summary.loc[roi_summary["focal_mode"] == "auto_median_2985_761"].copy()
    best_row = auto_summary.sort_values(["spearman", "mae_m"], ascending=[False, True]).iloc[0]
    best_column = str(best_row["roi_statistic"])
    best_estimate = roi_diagnostic[best_column]*(AUTO_FOCAL_MEDIAN_PX/FIXED_FOCAL_PX)

    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].bar(fixed_summary["roi_statistic"], fixed_summary["spearman"])
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set(title="Correlação de cada estatística de ROI com a referência", ylabel="Spearman")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", alpha=.25)

    axes[1].bar(auto_summary["roi_statistic"], auto_summary["mae_m"], color="tab:orange")
    axes[1].set(title="MAE com focal automática mediana", ylabel="MAE (m)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].grid(axis="y", alpha=.25)

    axes[2].scatter(roi_diagnostic["reference_distance_nominal_m"], best_estimate, c=roi_diagnostic["video_time_s"], cmap="viridis", s=35)
    max_axis = float(max(roi_diagnostic["reference_distance_nominal_m"].max(), best_estimate.max()))
    axes[2].plot([0, max_axis], [0, max_axis], "k--", label="Ideal")
    axes[2].set(title=f"Melhor estatística por correlação: {best_column}", xlabel="Referência (m)", ylabel="Estimativa (m)")
    axes[2].grid(alpha=.25)
    axes[2].legend()
    fig.savefig(ROI_DIAGNOSTIC_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Quadros analisados: {len(roi_diagnostic)}")
    print(f"Melhor estatística por correlação: {best_column}")
    display(roi_summary.sort_values(["focal_mode", "spearman", "mae_m"], ascending=[True, False, True]).round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Diagnóstico amostral de ROI bloqueado.")


## 8.4. Piloto de inferência em recortes centralizados

Execute somente após a seção 8.3. Na célula complementar da seção 1, altere `RUN_CROP_DEPTH_DIAGNOSTIC = True`. O teste reutiliza os mesmos 60 quadros e executa o Depth Pro em recortes quadrados de 512 e 1024 pixels centralizados no drone. Como o recorte não é redimensionado manualmente antes do `transform`, a focal permanece expressa no sistema de pixels original. Esta etapa faz até 120 inferências, permite retomada e deve levar aproximadamente 8–12 minutos.


In [ ]:
def centered_square_crop(frame: np.ndarray, box, requested_side: int):
    height, width = frame.shape[:2]
    side = int(min(requested_side, height, width))
    x1, y1, x2, y2 = map(float, box)
    center_x = (x1+x2)/2
    center_y = (y1+y2)/2
    crop_x1 = int(np.clip(round(center_x-side/2), 0, width-side))
    crop_y1 = int(np.clip(round(center_y-side/2), 0, height-side))
    crop = frame[crop_y1:crop_y1+side, crop_x1:crop_x1+side]
    adjusted_box = [x1-crop_x1, y1-crop_y1, x2-crop_x1, y2-crop_y1]
    return crop, adjusted_box, crop_x1, crop_y1


if RUN_CROP_DEPTH_DIAGNOSTIC:
    if not RUN_ROI_DIAGNOSTIC and "roi_diagnostic" not in globals():
        raise RuntimeError("Execute primeiro a seção 8.3 ou mantenha RUN_ROI_DIAGNOSTIC=True.")
    ROI_DIAGNOSTIC_CSV = OUTPUT_DIR / "roi_depth_diagnostic_frames.csv"
    if not ROI_DIAGNOSTIC_CSV.exists():
        raise FileNotFoundError("Execute primeiro a seção 8.3.")
    roi_diagnostic = pd.read_csv(ROI_DIAGNOSTIC_CSV)
    selected_ids = set(roi_diagnostic["frame_idx"].astype(int))
    selected = (
        depth_frames.loc[depth_frames["frame_idx"].astype(int).isin(selected_ids)]
        .drop_duplicates("frame_idx")
        .sort_values("frame_idx")
    )

    CROP_DEPTH_CSV = OUTPUT_DIR / "crop_depth_diagnostic_frames.csv"
    CROP_DEPTH_SUMMARY_CSV = OUTPUT_DIR / "crop_depth_diagnostic_summary.csv"
    CROP_DEPTH_PLOT = OUTPUT_DIR / "crop_depth_diagnostic.png"
    if CROP_DEPTH_CSV.exists() and not FORCE_RESTART:
        crop_records_df = pd.read_csv(CROP_DEPTH_CSV).drop_duplicates(
            ["frame_idx", "crop_side_px"], keep="last"
        )
    else:
        crop_records_df = pd.DataFrame()
    completed_pairs = set(
        zip(
            crop_records_df.get("frame_idx", pd.Series(dtype=int)).astype(int),
            crop_records_df.get("crop_side_px", pd.Series(dtype=int)).astype(int),
        )
    )
    pending_pairs = [
        (int(row["frame_idx"]), int(side))
        for _, row in selected.iterrows()
        for side in CROP_DIAGNOSTIC_SIDES_PX
        if (int(row["frame_idx"]), int(side)) not in completed_pairs
    ]

    if pending_pairs:
        model, transform, device, precision = load_depth_model()
        cap = cv2.VideoCapture(str(VIDEO_PATH))
        new_records = []
        total_pairs = len(selected)*len(CROP_DIAGNOSTIC_SIDES_PX)
        for _, row in selected.iterrows():
            frame_idx = int(row["frame_idx"])
            missing_sides = [
                int(side) for side in CROP_DIAGNOSTIC_SIDES_PX
                if (frame_idx, int(side)) not in completed_pairs
            ]
            if not missing_sides:
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, frame = cap.read()
            if not ok:
                print(f"Falha ao ler quadro {frame_idx}")
                continue
            original_box = parse_box(row)
            for requested_side in missing_sides:
                crop, crop_box, crop_x1, crop_y1 = centered_square_crop(frame, original_box, requested_side)
                pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).convert("RGB")
                tensor = transform(pil).to(device=device, dtype=precision)
                focal = torch.tensor([AUTO_FOCAL_MEDIAN_PX], device=device, dtype=tensor.dtype)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                started = time.perf_counter()
                with torch.inference_mode():
                    prediction = model.infer(tensor, f_px=focal)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                elapsed = time.perf_counter()-started
                depth = prediction["depth"].detach().float().cpu().numpy().squeeze()
                stats = roi_percentile_set(depth, crop_box, pil.size, 0.0)
                record = {
                    "frame_idx": frame_idx,
                    "video_time_s": float(row["video_time_s"]),
                    "reference_distance_nominal_m": float(row["reference_distance_nominal_m"]),
                    "distance_zone": row["distance_zone"],
                    "confidence": float(row["confidence"]),
                    "crop_side_px": int(crop.shape[1]),
                    "crop_x1": crop_x1,
                    "crop_y1": crop_y1,
                    "roi_pixels": stats["pixels"],
                    "depth_time_s": elapsed,
                }
                for q in [5, 10, 25, 50]:
                    record[f"p{q:02d}_m"] = stats[f"p{q:02d}"]
                new_records.append(record)
                combined = pd.concat([crop_records_df, pd.DataFrame(new_records)], ignore_index=True)
                combined = combined.drop_duplicates(["frame_idx", "crop_side_px"], keep="last")
                atomic_csv(combined.sort_values(["frame_idx", "crop_side_px"]), CROP_DEPTH_CSV)
                print(f"Recortes Depth: {len(completed_pairs)+len(new_records)}/{total_pairs}")
        cap.release()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    crop_diagnostic = pd.read_csv(CROP_DEPTH_CSV)
    crop_summary_rows = []
    for side, part in crop_diagnostic.groupby("crop_side_px"):
        for q in [5, 10, 25, 50]:
            estimate_column = f"p{q:02d}_m"
            reference = part["reference_distance_nominal_m"]
            estimate = part[estimate_column]
            crop_summary_rows.append({
                "input_region": f"crop_{int(side)}px",
                "roi_statistic": estimate_column,
                "n": int(len(part)),
                "mean_estimate_m": float(estimate.mean()),
                "median_estimate_m": float(estimate.median()),
                "saturation_10000_pct": float((estimate >= 9999.0).mean()*100),
                **metric_dict(reference, estimate),
                **acceptance_dict(reference, estimate),
            })
    for q in [5, 10, 25, 50]:
        estimate_column = f"full_p{q:02d}_m"
        reference = roi_diagnostic["reference_distance_nominal_m"]
        estimate = roi_diagnostic[estimate_column]*(AUTO_FOCAL_MEDIAN_PX/FIXED_FOCAL_PX)
        crop_summary_rows.append({
            "input_region": "full_frame",
            "roi_statistic": f"p{q:02d}_m",
            "n": int(len(roi_diagnostic)),
            "mean_estimate_m": float(estimate.mean()),
            "median_estimate_m": float(estimate.median()),
            "saturation_10000_pct": float((estimate >= 9999.0).mean()*100),
            **metric_dict(reference, estimate),
            **acceptance_dict(reference, estimate),
        })
    crop_summary = pd.DataFrame(crop_summary_rows)
    atomic_csv(crop_summary, CROP_DEPTH_SUMMARY_CSV)
    top_rank = crop_summary.sort_values(["spearman", "mae_m"], ascending=[False, True]).iloc[0]
    top_region = str(top_rank["input_region"])
    top_stat = str(top_rank["roi_statistic"])

    labels = crop_summary["input_region"] + "_" + crop_summary["roi_statistic"]
    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].bar(labels, crop_summary["spearman"])
    axes[0].set(title="Correlação: quadro completo versus recortes", ylabel="Spearman")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", alpha=.25)
    axes[1].bar(labels, crop_summary["mae_m"], color="tab:orange")
    axes[1].set(title="MAE: quadro completo versus recortes", ylabel="MAE (m)")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].grid(axis="y", alpha=.25)

    if top_region == "full_frame":
        q = int(top_stat[1:3])
        top_reference = roi_diagnostic["reference_distance_nominal_m"]
        top_estimate = roi_diagnostic[f"full_p{q:02d}_m"]*(AUTO_FOCAL_MEDIAN_PX/FIXED_FOCAL_PX)
    else:
        side = int(top_region.split("_")[1].replace("px", ""))
        top_part = crop_diagnostic.loc[crop_diagnostic["crop_side_px"] == side]
        top_reference = top_part["reference_distance_nominal_m"]
        top_estimate = top_part[top_stat]
    min_axis = float(max(0.1, min(top_reference.min(), top_estimate.min())))
    max_axis = float(max(top_reference.max(), top_estimate.max()))
    axes[2].scatter(top_reference, top_estimate, s=35, alpha=.75)
    axes[2].plot([min_axis, max_axis], [min_axis, max_axis], "k--", label="Ideal")
    axes[2].set_xscale("log")
    axes[2].set_yscale("log")
    axes[2].set(title=f"Maior correlação monotônica: {top_region} / {top_stat}", xlabel="Referência (m)", ylabel="Estimativa bruta (m)")
    axes[2].grid(alpha=.25)
    axes[2].legend()
    fig.savefig(CROP_DEPTH_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Maior Spearman (não significa melhor precisão): {top_region} / {top_stat}")
    print("Recortes são avaliados como sinais relativos; não use suas saídas brutas como metros.")
    display(crop_summary.sort_values(["spearman", "mae_m"], ascending=[False, True]).round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Diagnóstico Depth em recortes bloqueado.")


## 8.5. Calibração exploratória com validação bloqueada no tempo

Execute após a seção 8.4, alterando `RUN_CROP_CALIBRATION_CV = True`. Esta etapa **não executa o Depth Pro novamente**: ela reutiliza os CSVs das seções 8.3 e 8.4.

O valor bruto do recorte de 1024 px não é interpretado como distância métrica. Ele entra apenas como uma variável relativa, após transformação logarítmica. A avaliação usa predições fora da amostra: em cada rodada, um bloco temporal de 10 segundos fica totalmente fora do ajuste. O modelo híbrido acrescenta o quadro completo e a área da caixa YOLO.

Esta continua sendo uma validação interna exploratória em apenas 60 quadros, balanceados por faixa de distância. Ela serve para decidir se o sinal merece uma etapa posterior de calibração, mas não substitui teste em outro voo/vídeo.


In [ ]:
if RUN_CROP_CALIBRATION_CV:
    ROI_DIAGNOSTIC_CSV = OUTPUT_DIR / "roi_depth_diagnostic_frames.csv"
    CROP_DEPTH_CSV = OUTPUT_DIR / "crop_depth_diagnostic_frames.csv"
    for required_path in [ROI_DIAGNOSTIC_CSV, CROP_DEPTH_CSV]:
        if not required_path.exists():
            raise FileNotFoundError(f"Arquivo ausente: {required_path}. Execute as seções 8.3 e 8.4.")

    roi_cv = pd.read_csv(ROI_DIAGNOSTIC_CSV).drop_duplicates("frame_idx", keep="last")
    crop_cv = pd.read_csv(CROP_DEPTH_CSV).drop_duplicates(
        ["frame_idx", "crop_side_px"], keep="last"
    )
    crop_512 = (
        crop_cv.loc[crop_cv["crop_side_px"].astype(int) == 512, ["frame_idx", "p05_m"]]
        .rename(columns={"p05_m": "crop_512_p05_m"})
    )
    crop_1024 = (
        crop_cv.loc[crop_cv["crop_side_px"].astype(int) == 1024, ["frame_idx", "p10_m"]]
        .rename(columns={"p10_m": "crop_1024_p10_m"})
    )
    cv_data = (
        roi_cv.merge(crop_512, on="frame_idx", how="inner")
        .merge(crop_1024, on="frame_idx", how="inner")
        .sort_values("frame_idx")
        .reset_index(drop=True)
    )
    cv_data["box_area_px2"] = cv_data["box_width_px"]*cv_data["box_height_px"]
    cv_data["time_block_10s"] = np.floor(cv_data["video_time_s"]/10).astype(int)

    candidate_features = {
        "full_p05_log": ["full_p05_m"],
        "crop_512_p05_log": ["crop_512_p05_m"],
        "crop_1024_p10_log": ["crop_1024_p10_m"],
        "box_area_log": ["box_area_px2"],
        "hybrid_crop_full_box_log": [
            "crop_1024_p10_m", "crop_512_p05_m", "full_p05_m", "box_area_px2"
        ],
    }
    needed = sorted({column for columns in candidate_features.values() for column in columns})
    numeric_columns = needed + ["reference_distance_nominal_m", "video_time_s"]
    for column in numeric_columns:
        cv_data[column] = pd.to_numeric(cv_data[column], errors="coerce")
    finite_positive = np.ones(len(cv_data), dtype=bool)
    for column in needed + ["reference_distance_nominal_m"]:
        finite_positive &= np.isfinite(cv_data[column]) & (cv_data[column] > 0)
    cv_data = cv_data.loc[finite_positive].copy().reset_index(drop=True)
    if len(cv_data) < 20 or cv_data["time_block_10s"].nunique() < 3:
        raise RuntimeError("Amostra insuficiente para validação temporal bloqueada.")

    def fit_log_ridge(train, feature_columns, alpha=1.0):
        x = np.log(np.maximum(train[feature_columns].to_numpy(float), 1e-9))
        center = x.mean(axis=0)
        scale = x.std(axis=0, ddof=0)
        scale[~np.isfinite(scale) | (scale < 1e-9)] = 1.0
        z = (x-center)/scale
        design = np.column_stack([np.ones(len(z)), z])
        target = np.log(np.maximum(train["reference_distance_nominal_m"].to_numpy(float), 1e-9))
        penalty = np.eye(design.shape[1])
        penalty[0, 0] = 0.0
        beta = np.linalg.pinv(design.T@design + alpha*penalty)@(design.T@target)
        return center, scale, beta

    def predict_log_ridge(data, feature_columns, fitted):
        center, scale, beta = fitted
        x = np.log(np.maximum(data[feature_columns].to_numpy(float), 1e-9))
        z = (x-center)/scale
        log_prediction = np.column_stack([np.ones(len(z)), z])@beta
        return np.exp(np.clip(log_prediction, -20, 20))

    groups = sorted(cv_data["time_block_10s"].unique())
    prediction_rows = []
    for model_name, feature_columns in candidate_features.items():
        oof = np.full(len(cv_data), np.nan)
        for held_out in groups:
            train_mask = cv_data["time_block_10s"] != held_out
            test_mask = ~train_mask
            if train_mask.sum() <= len(feature_columns)+2:
                continue
            fitted = fit_log_ridge(cv_data.loc[train_mask], feature_columns)
            oof[test_mask] = predict_log_ridge(cv_data.loc[test_mask], feature_columns, fitted)
        for index, prediction in enumerate(oof):
            prediction_rows.append({
                "frame_idx": int(cv_data.loc[index, "frame_idx"]),
                "video_time_s": float(cv_data.loc[index, "video_time_s"]),
                "time_block_10s": int(cv_data.loc[index, "time_block_10s"]),
                "distance_zone": cv_data.loc[index, "distance_zone"],
                "reference_distance_nominal_m": float(cv_data.loc[index, "reference_distance_nominal_m"]),
                "model": model_name,
                "oof_estimate_m": float(prediction),
            })

    cv_predictions = pd.DataFrame(prediction_rows)
    summary_rows = []
    for model_name, part in cv_predictions.groupby("model", sort=False):
        reference = part["reference_distance_nominal_m"]
        estimate = part["oof_estimate_m"]
        summary_rows.append({
            "model": model_name,
            "validation": "leave_one_10s_block_out",
            "features": ", ".join(candidate_features[model_name]),
            **metric_dict(reference, estimate),
            **acceptance_dict(reference, estimate),
        })
    cv_summary = pd.DataFrame(summary_rows).sort_values(["rmse_m", "mae_m"])

    zone_rows = []
    for (model_name, zone), part in cv_predictions.groupby(["model", "distance_zone"]):
        zone_rows.append({
            "model": model_name,
            "distance_zone": zone,
            **metric_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
            **acceptance_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
        })
    cv_zones = pd.DataFrame(zone_rows)

    coefficient_rows = []
    for model_name, feature_columns in candidate_features.items():
        center, scale, beta = fit_log_ridge(cv_data, feature_columns)
        coefficient_rows.append({
            "model": model_name, "term": "intercept_log_m",
            "coefficient_standardized": float(beta[0]),
        })
        for feature, coefficient, mean_log, scale_log in zip(feature_columns, beta[1:], center, scale):
            coefficient_rows.append({
                "model": model_name, "term": feature,
                "coefficient_standardized": float(coefficient),
                "training_log_mean": float(mean_log),
                "training_log_scale": float(scale_log),
            })
    cv_coefficients = pd.DataFrame(coefficient_rows)

    CV_PREDICTIONS_CSV = OUTPUT_DIR / "crop_calibration_blocked_cv_predictions.csv"
    CV_SUMMARY_CSV = OUTPUT_DIR / "crop_calibration_blocked_cv_summary.csv"
    CV_ZONES_CSV = OUTPUT_DIR / "crop_calibration_blocked_cv_zones.csv"
    CV_COEFFICIENTS_CSV = OUTPUT_DIR / "crop_calibration_provisional_coefficients.csv"
    CV_PLOT = OUTPUT_DIR / "crop_calibration_blocked_cv.png"
    atomic_csv(cv_predictions, CV_PREDICTIONS_CSV)
    atomic_csv(cv_summary, CV_SUMMARY_CSV)
    atomic_csv(cv_zones, CV_ZONES_CSV)
    atomic_csv(cv_coefficients, CV_COEFFICIENTS_CSV)

    best_model = str(cv_summary.iloc[0]["model"])
    best_predictions = cv_predictions.loc[cv_predictions["model"] == best_model].sort_values("video_time_s")
    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].bar(cv_summary["model"], cv_summary["mae_m"], label="MAE")
    axes[0].scatter(cv_summary["model"], cv_summary["rmse_m"], color="tab:red", label="RMSE", zorder=3)
    axes[0].set(title="Validação bloqueada: erro fora da amostra", ylabel="Erro (m)")
    axes[0].tick_params(axis="x", rotation=25)
    axes[0].grid(axis="y", alpha=.25)
    axes[0].legend()

    reference = best_predictions["reference_distance_nominal_m"]
    estimate = best_predictions["oof_estimate_m"]
    axis_max = float(max(reference.max(), estimate.max()))
    axes[1].scatter(reference, estimate, c=best_predictions["video_time_s"], cmap="viridis", s=45)
    axes[1].plot([0, axis_max], [0, axis_max], "k--", label="Ideal")
    axes[1].set(title=f"Melhor RMSE bloqueado: {best_model}", xlabel="Referência (m)", ylabel="Predição OOF (m)")
    axes[1].grid(alpha=.25)
    axes[1].legend()

    axes[2].plot(best_predictions["video_time_s"], reference, color="black", linewidth=2, label="Referência")
    axes[2].plot(best_predictions["video_time_s"], estimate, marker="o", label="Predição OOF")
    axes[2].set(title="Predições fora da amostra ao longo do vídeo", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[2].grid(alpha=.25)
    axes[2].legend()
    fig.savefig(CV_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Quadros na validação: {len(cv_data)}; blocos temporais: {len(groups)}")
    print(f"Melhor RMSE interno bloqueado: {best_model}")
    print("Resultado exploratório: não promover coeficientes para produção sem teste em outro voo/vídeo.")
    display(cv_summary.round(3))
    display(cv_zones.loc[cv_zones["model"] == best_model].round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Calibração exploratória dos recortes bloqueada.")


## 8.6. Validação integral da distância pela caixa YOLO

Execute após a seção 8.5, alterando `RUN_FULL_BOX_CALIBRATION_CV = True`. A etapa **não executa YOLO nem Depth Pro**: ela reutiliza `depth_all_frames.csv`, seleciona caixas YOLO confiáveis e agrega as variáveis por segundo antes da validação.

As predições são sempre fora da amostra. Cada rodada retira um bloco temporal completo de 20 segundos do ajuste, reduzindo a dependência entre quadros vizinhos. São comparados largura, altura, área, formato da caixa, profundidade bruta e combinações entre esses sinais. Também é incluído o modelo geométrico restrito `distância ∝ área⁻⁰·⁵`, esperado aproximadamente para um objeto de tamanho constante sob projeção pinhole.

Mesmo esta validação continua pertencendo ao mesmo voo. Os coeficientes resultantes são provisórios até serem testados em outro vídeo.


In [ ]:
if RUN_FULL_BOX_CALIBRATION_CV:
    if not len(depth_frames):
        raise RuntimeError("Carregue depth_all_frames.csv antes da seção 8.6.")

    full_box = depth_frames.copy()
    required_columns = [
        "frame_idx", "video_time_s", "reference_distance_nominal_m",
        "confidence", "x1", "y1", "x2", "y2", "raw_s00_p25_m",
    ]
    missing = [column for column in required_columns if column not in full_box.columns]
    if missing:
        raise KeyError(f"Colunas ausentes para a validação integral: {missing}")
    for column in required_columns:
        full_box[column] = pd.to_numeric(full_box[column], errors="coerce")

    full_box["box_width_px"] = full_box["x2"]-full_box["x1"]
    full_box["box_height_px"] = full_box["y2"]-full_box["y1"]
    full_box["box_area_px2"] = full_box["box_width_px"]*full_box["box_height_px"]
    full_box["box_aspect_ratio"] = full_box["box_width_px"]/full_box["box_height_px"]
    full_box["second"] = np.floor(full_box["video_time_s"]).astype("Int64")

    labeled_frame = np.isfinite(full_box["reference_distance_nominal_m"])
    reliable_box = (
        labeled_frame
        & np.isfinite(full_box["confidence"])
        & (full_box["confidence"] >= YOLO_RELIABLE_CONF)
        & np.isfinite(full_box[["box_width_px", "box_height_px", "box_area_px2"]]).all(axis=1)
        & (full_box["box_width_px"] >= MIN_BOX_WIDTH_PX)
        & (full_box["box_height_px"] >= MIN_BOX_HEIGHT_PX)
        & (full_box["box_area_px2"] > 0)
    )
    reliable = full_box.loc[reliable_box].copy()

    total_labeled_seconds = int(full_box.loc[labeled_frame, "second"].nunique())
    second_data = reliable.groupby("second", as_index=False).agg(
        video_time_s=("video_time_s", "median"),
        reference_distance_nominal_m=("reference_distance_nominal_m", "median"),
        box_width_px=("box_width_px", "median"),
        box_height_px=("box_height_px", "median"),
        box_area_px2=("box_area_px2", "median"),
        box_aspect_ratio=("box_aspect_ratio", "median"),
        confidence=("confidence", "median"),
        raw_s00_p25_m=("raw_s00_p25_m", "median"),
        reliable_frames=("frame_idx", "size"),
    )
    second_data["distance_zone"] = second_data["reference_distance_nominal_m"].map(distance_zone)
    second_data["time_block_20s"] = np.floor(second_data["video_time_s"]/20).astype(int)

    feature_columns = [
        "box_width_px", "box_height_px", "box_area_px2",
        "box_aspect_ratio", "confidence", "raw_s00_p25_m",
    ]
    valid_seconds = np.ones(len(second_data), dtype=bool)
    for column in feature_columns + ["reference_distance_nominal_m"]:
        valid_seconds &= np.isfinite(second_data[column]) & (second_data[column] > 0)
    second_data = second_data.loc[valid_seconds].copy().reset_index(drop=True)
    if len(second_data) < 20 or second_data["time_block_20s"].nunique() < 3:
        raise RuntimeError("Segundos válidos insuficientes para validação temporal integral.")

    model_specs = {
        "box_width_log": {"features": ["box_width_px"]},
        "box_height_log": {"features": ["box_height_px"]},
        "box_area_log": {"features": ["box_area_px2"]},
        "box_width_height_log": {"features": ["box_width_px", "box_height_px"]},
        "box_area_aspect_log": {"features": ["box_area_px2", "box_aspect_ratio"]},
        "depth_p25_log": {"features": ["raw_s00_p25_m"]},
        "box_area_depth_log": {"features": ["box_area_px2", "raw_s00_p25_m"]},
        "geometry_area_power_-0.5": {"features": ["box_area_px2"], "fixed_slopes": [-0.5]},
    }

    def fit_full_log_model(train, specification, alpha=1.0):
        features = specification["features"]
        x_log = np.log(np.maximum(train[features].to_numpy(float), 1e-9))
        y_log = np.log(np.maximum(train["reference_distance_nominal_m"].to_numpy(float), 1e-9))
        if "fixed_slopes" in specification:
            slopes = np.asarray(specification["fixed_slopes"], float)
            intercept = float(np.mean(y_log-x_log@slopes))
            return {"kind": "fixed", "intercept": intercept, "slopes": slopes}
        center = x_log.mean(axis=0)
        scale = x_log.std(axis=0, ddof=0)
        scale[~np.isfinite(scale) | (scale < 1e-9)] = 1.0
        z = (x_log-center)/scale
        design = np.column_stack([np.ones(len(z)), z])
        penalty = np.eye(design.shape[1])
        penalty[0, 0] = 0.0
        beta = np.linalg.pinv(design.T@design + alpha*penalty)@(design.T@y_log)
        return {"kind": "ridge", "center": center, "scale": scale, "beta": beta}

    def predict_full_log_model(data, specification, fitted):
        features = specification["features"]
        x_log = np.log(np.maximum(data[features].to_numpy(float), 1e-9))
        if fitted["kind"] == "fixed":
            prediction_log = fitted["intercept"]+x_log@fitted["slopes"]
        else:
            z = (x_log-fitted["center"])/fitted["scale"]
            prediction_log = np.column_stack([np.ones(len(z)), z])@fitted["beta"]
        return np.exp(np.clip(prediction_log, -20, 20))

    groups = sorted(second_data["time_block_20s"].unique())
    prediction_rows = []
    for model_name, specification in model_specs.items():
        oof = np.full(len(second_data), np.nan)
        for held_out in groups:
            train_mask = second_data["time_block_20s"] != held_out
            test_mask = ~train_mask
            if train_mask.sum() <= len(specification["features"])+2:
                continue
            fitted = fit_full_log_model(second_data.loc[train_mask], specification)
            oof[test_mask] = predict_full_log_model(second_data.loc[test_mask], specification, fitted)
        for index, prediction in enumerate(oof):
            prediction_rows.append({
                "second": int(second_data.loc[index, "second"]),
                "video_time_s": float(second_data.loc[index, "video_time_s"]),
                "time_block_20s": int(second_data.loc[index, "time_block_20s"]),
                "distance_zone": second_data.loc[index, "distance_zone"],
                "reference_distance_nominal_m": float(second_data.loc[index, "reference_distance_nominal_m"]),
                "reliable_frames": int(second_data.loc[index, "reliable_frames"]),
                "model": model_name,
                "oof_estimate_m": float(prediction),
            })
    full_cv_predictions = pd.DataFrame(prediction_rows)

    summary_rows = []
    second_coverage = float(len(second_data)/max(1, total_labeled_seconds)*100)
    frame_coverage = float(reliable_box.sum()/max(1, labeled_frame.sum())*100)
    for model_name, part in full_cv_predictions.groupby("model", sort=False):
        summary_rows.append({
            "model": model_name,
            "validation": "leave_one_20s_block_out",
            "features": ", ".join(model_specs[model_name]["features"]),
            "seconds": int(len(part)),
            "second_coverage_pct": second_coverage,
            "reliable_frame_coverage_pct": frame_coverage,
            **metric_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
            **acceptance_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
        })
    full_cv_summary = pd.DataFrame(summary_rows).sort_values(["rmse_m", "mae_m"])

    zone_rows = []
    for (model_name, zone), part in full_cv_predictions.groupby(["model", "distance_zone"]):
        zone_rows.append({
            "model": model_name,
            "distance_zone": zone,
            "seconds": int(len(part)),
            **metric_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
            **acceptance_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
        })
    full_cv_zones = pd.DataFrame(zone_rows)

    coefficient_rows = []
    for model_name, specification in model_specs.items():
        fitted = fit_full_log_model(second_data, specification)
        if fitted["kind"] == "fixed":
            coefficient_rows.append({"model": model_name, "term": "intercept_log_m", "coefficient": fitted["intercept"]})
            for feature, coefficient in zip(specification["features"], fitted["slopes"]):
                coefficient_rows.append({"model": model_name, "term": feature, "coefficient": float(coefficient)})
        else:
            beta = fitted["beta"]
            slopes_original = beta[1:]/fitted["scale"]
            intercept_original = beta[0]-np.sum(slopes_original*fitted["center"])
            coefficient_rows.append({"model": model_name, "term": "intercept_log_m", "coefficient": float(intercept_original)})
            for feature, coefficient in zip(specification["features"], slopes_original):
                coefficient_rows.append({"model": model_name, "term": feature, "coefficient": float(coefficient)})
    full_cv_coefficients = pd.DataFrame(coefficient_rows)

    FULL_BOX_SECOND_CSV = OUTPUT_DIR / "box_distance_per_second_inputs.csv"
    FULL_BOX_PREDICTIONS_CSV = OUTPUT_DIR / "box_distance_blocked_cv_predictions.csv"
    FULL_BOX_SUMMARY_CSV = OUTPUT_DIR / "box_distance_blocked_cv_summary.csv"
    FULL_BOX_ZONES_CSV = OUTPUT_DIR / "box_distance_blocked_cv_zones.csv"
    FULL_BOX_COEFFICIENTS_CSV = OUTPUT_DIR / "box_distance_provisional_coefficients.csv"
    FULL_BOX_PLOT = OUTPUT_DIR / "box_distance_blocked_cv.png"
    atomic_csv(second_data, FULL_BOX_SECOND_CSV)
    atomic_csv(full_cv_predictions, FULL_BOX_PREDICTIONS_CSV)
    atomic_csv(full_cv_summary, FULL_BOX_SUMMARY_CSV)
    atomic_csv(full_cv_zones, FULL_BOX_ZONES_CSV)
    atomic_csv(full_cv_coefficients, FULL_BOX_COEFFICIENTS_CSV)

    best_model = str(full_cv_summary.iloc[0]["model"])
    best_predictions = full_cv_predictions.loc[
        full_cv_predictions["model"] == best_model
    ].sort_values("video_time_s")
    best_zones = full_cv_zones.loc[full_cv_zones["model"] == best_model]

    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].bar(full_cv_summary["model"], full_cv_summary["mae_m"], label="MAE")
    axes[0].scatter(full_cv_summary["model"], full_cv_summary["rmse_m"], color="tab:red", label="RMSE", zorder=3)
    axes[0].set(title="Sequência completa: erro por segundo fora da amostra", ylabel="Erro (m)")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].grid(axis="y", alpha=.25)
    axes[0].legend()

    reference = best_predictions["reference_distance_nominal_m"]
    estimate = best_predictions["oof_estimate_m"]
    axis_max = float(max(reference.max(), estimate.max()))
    axes[1].scatter(reference, estimate, c=best_predictions["video_time_s"], cmap="viridis", s=40)
    axes[1].plot([0, axis_max], [0, axis_max], "k--", label="Ideal")
    axes[1].set(title=f"Melhor RMSE integral: {best_model}", xlabel="Referência (m)", ylabel="Predição OOF (m)")
    axes[1].grid(alpha=.25)
    axes[1].legend()

    axes[2].plot(best_predictions["video_time_s"], reference, color="black", linewidth=2, label="Referência")
    axes[2].plot(best_predictions["video_time_s"], estimate, color="tab:blue", marker="o", markersize=3, label="Predição OOF")
    axes[2].set(title="Sequência integral agregada por segundo", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[2].grid(alpha=.25)
    axes[2].legend()
    fig.savefig(FULL_BOX_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Segundos rotulados/válidos: {total_labeled_seconds}/{len(second_data)}")
    print(f"Cobertura de caixas confiáveis por quadro: {frame_coverage:.3f}%")
    print(f"Blocos temporais de 20 s: {len(groups)}")
    print(f"Melhor RMSE integral bloqueado: {best_model}")
    print("Coeficientes permanecem provisórios até validação em outro vídeo.")
    display(full_cv_summary.round(3))
    display(best_zones.round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Validação integral pela caixa YOLO bloqueada.")


## 8.7. Auditoria de robustez e equação provisória

Execute após a seção 8.6, alterando `RUN_BOX_ROBUSTNESS_AUDIT = True`. Esta etapa não executa inferência. Ela audita o modelo vencedor `box_width_log`, identifica os maiores erros e avalia uma regra operacional de abstenção que não consulta a distância de referência.

A regra exige pelo menos cinco caixas confiáveis no segundo, limita a dispersão relativa da largura e aplica um teste causal de Hampel ao logaritmo da largura usando apenas os cinco segundos anteriores. Os limiares são definidos antes da avaliação e não são otimizados pelo erro observado.

A versão sem abstenção permanece como resultado científico primário. A versão com controle de qualidade é uma análise operacional secundária. Ao final, a célula salva a equação provisória no formato `distância = constante × largura^expoente`.


In [ ]:
if RUN_BOX_ROBUSTNESS_AUDIT:
    SECOND_INPUT_CSV = OUTPUT_DIR / "box_distance_per_second_inputs.csv"
    BLOCKED_PREDICTIONS_CSV = OUTPUT_DIR / "box_distance_blocked_cv_predictions.csv"
    COEFFICIENTS_CSV = OUTPUT_DIR / "box_distance_provisional_coefficients.csv"
    for required_path in [SECOND_INPUT_CSV, BLOCKED_PREDICTIONS_CSV, COEFFICIENTS_CSV]:
        if not required_path.exists():
            raise FileNotFoundError(f"Arquivo ausente: {required_path}. Execute primeiro a seção 8.6.")

    # Reconstrói estatísticas de dispersão dentro de cada segundo.
    audit_frames = depth_frames.copy()
    numeric = [
        "frame_idx", "video_time_s", "reference_distance_nominal_m",
        "confidence", "x1", "y1", "x2", "y2",
    ]
    for column in numeric:
        audit_frames[column] = pd.to_numeric(audit_frames[column], errors="coerce")
    audit_frames["box_width_px"] = audit_frames["x2"]-audit_frames["x1"]
    audit_frames["box_height_px"] = audit_frames["y2"]-audit_frames["y1"]
    audit_frames["second"] = np.floor(audit_frames["video_time_s"]).astype("Int64")
    labeled = np.isfinite(audit_frames["reference_distance_nominal_m"])
    reliable = (
        labeled
        & np.isfinite(audit_frames["confidence"])
        & (audit_frames["confidence"] >= YOLO_RELIABLE_CONF)
        & np.isfinite(audit_frames[["box_width_px", "box_height_px"]]).all(axis=1)
        & (audit_frames["box_width_px"] >= MIN_BOX_WIDTH_PX)
        & (audit_frames["box_height_px"] >= MIN_BOX_HEIGHT_PX)
    )
    reliable_frames = audit_frames.loc[reliable].copy()

    def percentile_25(values):
        return float(np.percentile(values, 25))

    def percentile_75(values):
        return float(np.percentile(values, 75))

    def median_absolute_deviation(values):
        values = np.asarray(values, float)
        median = np.median(values)
        return float(np.median(np.abs(values-median)))

    second_quality = reliable_frames.groupby("second", as_index=False).agg(
        video_time_s=("video_time_s", "median"),
        reference_distance_nominal_m=("reference_distance_nominal_m", "median"),
        width_median_px=("box_width_px", "median"),
        width_q25_px=("box_width_px", percentile_25),
        width_q75_px=("box_width_px", percentile_75),
        width_mad_px=("box_width_px", median_absolute_deviation),
        confidence_median=("confidence", "median"),
        reliable_frames=("frame_idx", "size"),
    )
    second_quality["width_relative_iqr"] = (
        (second_quality["width_q75_px"]-second_quality["width_q25_px"])
        /second_quality["width_median_px"].clip(lower=1e-9)
    )
    second_quality["distance_zone"] = second_quality["reference_distance_nominal_m"].map(distance_zone)

    predictions = pd.read_csv(BLOCKED_PREDICTIONS_CSV)
    predictions = predictions.loc[predictions["model"] == "box_width_log"].copy()
    predictions["second"] = pd.to_numeric(predictions["second"], errors="coerce").astype("Int64")
    audit = (
        second_quality.merge(
            predictions[["second", "time_block_20s", "oof_estimate_m"]],
            on="second", how="inner",
        )
        .sort_values("second")
        .reset_index(drop=True)
    )
    audit["absolute_error_m"] = np.abs(
        audit["oof_estimate_m"]-audit["reference_distance_nominal_m"]
    )
    audit["signed_error_m"] = audit["oof_estimate_m"]-audit["reference_distance_nominal_m"]

    # Teste causal sobre a variação da largura: evita rejeitar tendências suaves.
    log_width = np.log(audit["width_median_px"].clip(lower=1e-9).to_numpy(float))
    seconds_array = audit["second"].to_numpy(float)
    log_delta = np.full(len(audit), np.nan)
    consecutive = np.diff(seconds_array) == 1
    log_delta[1:][consecutive] = np.diff(log_width)[consecutive]
    temporal_outlier = np.zeros(len(audit), dtype=bool)
    temporal_score = np.full(len(audit), np.nan)
    history_size = int(BOX_AUDIT_HAMPEL_HISTORY_SECONDS)
    for index in range(len(audit)):
        if not np.isfinite(log_delta[index]):
            continue
        current_second = int(audit.loc[index, "second"])
        history_mask = (
            (seconds_array < current_second)
            & (seconds_array >= current_second-history_size)
            & np.isfinite(log_delta)
        )
        history = log_delta[history_mask]
        if len(history) < 3:
            continue
        center = float(np.median(history))
        mad = float(np.median(np.abs(history-center)))
        robust_scale = 1.4826*mad
        residual = abs(log_delta[index]-center)
        threshold = max(
            BOX_AUDIT_HAMPEL_K*robust_scale,
            float(BOX_AUDIT_MAX_LOG_WIDTH_JUMP),
        )
        temporal_score[index] = residual/max(robust_scale, 1e-9)
        temporal_outlier[index] = residual > threshold
    audit["log_width_change"] = log_delta

    audit["temporal_hampel_score"] = temporal_score
    audit["temporal_width_outlier"] = temporal_outlier
    audit["low_support"] = audit["reliable_frames"] < BOX_AUDIT_MIN_FRAMES_PER_SECOND
    audit["unstable_within_second"] = audit["width_relative_iqr"] > BOX_AUDIT_MAX_RELATIVE_IQR
    audit["quality_abstained"] = (
        audit["low_support"]
        | audit["unstable_within_second"]
        | audit["temporal_width_outlier"]
    )

    def quality_reason(row):
        reasons = []
        if bool(row["low_support"]):
            reasons.append("few_reliable_frames")
        if bool(row["unstable_within_second"]):
            reasons.append("unstable_width_iqr")
        if bool(row["temporal_width_outlier"]):
            reasons.append("causal_hampel_outlier")
        return ";".join(reasons)

    audit["quality_reason"] = audit.apply(quality_reason, axis=1)
    total_labeled_seconds = int(audit_frames.loc[labeled, "second"].nunique())
    emitted_quality = audit.loc[~audit["quality_abstained"]].copy()

    comparison_rows = []
    for label, part in [
        ("box_width_log_oof_primary", audit),
        ("box_width_log_oof_quality_gated", emitted_quality),
    ]:
        comparison_rows.append({
            "variant": label,
            "seconds_emitted": int(len(part)),
            "coverage_all_labeled_seconds_pct": float(len(part)/max(1, total_labeled_seconds)*100),
            "coverage_box_valid_seconds_pct": float(len(part)/max(1, len(audit))*100),
            **metric_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
            **acceptance_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
        })
    quality_comparison = pd.DataFrame(comparison_rows)

    block_rows = []
    for block, block_all in audit.groupby("time_block_20s"):
        for label, part in [
            ("primary", block_all),
            ("quality_gated", block_all.loc[~block_all["quality_abstained"]]),
        ]:
            block_rows.append({
                "time_block_20s": int(block),
                "variant": label,
                "seconds_available": int(len(block_all)),
                "seconds_emitted": int(len(part)),
                **metric_dict(part["reference_distance_nominal_m"], part["oof_estimate_m"]),
            })
    block_metrics = pd.DataFrame(block_rows)

    coefficients = pd.read_csv(COEFFICIENTS_CSV)
    width_coefficients = coefficients.loc[coefficients["model"] == "box_width_log"].copy()
    intercept = float(width_coefficients.loc[
        width_coefficients["term"] == "intercept_log_m", "coefficient"
    ].iloc[0])
    exponent = float(width_coefficients.loc[
        width_coefficients["term"] == "box_width_px", "coefficient"
    ].iloc[0])
    constant = float(np.exp(intercept))
    model_specification = {
        "status": "provisional_internal_validation_only",
        "model": "box_width_log",
        "equation": "distance_m = constant * median_box_width_px ** exponent",
        "constant": constant,
        "exponent": exponent,
        "intercept_log_m": intercept,
        "input_resolution_px": [int(WIDTH), int(HEIGHT)],
        "aggregation": "median width of reliable YOLO boxes per second",
        "yolo_reliable_confidence": float(YOLO_RELIABLE_CONF),
        "minimum_box_width_px": float(MIN_BOX_WIDTH_PX),
        "minimum_box_height_px": float(MIN_BOX_HEIGHT_PX),
        "quality_gate": {
            "minimum_reliable_frames_per_second": int(BOX_AUDIT_MIN_FRAMES_PER_SECOND),
            "maximum_relative_width_iqr": float(BOX_AUDIT_MAX_RELATIVE_IQR),
            "causal_hampel_history_seconds": int(BOX_AUDIT_HAMPEL_HISTORY_SECONDS),
            "causal_hampel_k": float(BOX_AUDIT_HAMPEL_K),
            "minimum_log_width_jump": float(BOX_AUDIT_MAX_LOG_WIDTH_JUMP),
        },
        "limitations": [
            "single flight internal temporal validation",
            "fixed zoom required",
            "same drone appearance required",
            "only two valid seconds above 40 m",
        ],
    }

    AUDIT_CSV = OUTPUT_DIR / "box_width_robustness_audit.csv"
    QUALITY_COMPARISON_CSV = OUTPUT_DIR / "box_width_quality_comparison.csv"
    BLOCK_METRICS_CSV = OUTPUT_DIR / "box_width_block_metrics.csv"
    MODEL_JSON = OUTPUT_DIR / "box_width_model_provisional.json"
    AUDIT_PLOT = OUTPUT_DIR / "box_width_robustness_audit.png"
    atomic_csv(audit, AUDIT_CSV)
    atomic_csv(quality_comparison, QUALITY_COMPARISON_CSV)
    atomic_csv(block_metrics, BLOCK_METRICS_CSV)
    with MODEL_JSON.open("w", encoding="utf-8") as handle:
        json.dump(model_specification, handle, indent=2, ensure_ascii=False)

    top_errors = audit.sort_values("absolute_error_m", ascending=False).head(12)
    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].plot(audit["video_time_s"], audit["reference_distance_nominal_m"], color="black", linewidth=2, label="Referência")
    axes[0].plot(audit["video_time_s"], audit["oof_estimate_m"], color="tab:blue", marker="o", markersize=3, label="Predição OOF")
    rejected = audit.loc[audit["quality_abstained"]]
    axes[0].scatter(rejected["video_time_s"], rejected["oof_estimate_m"], color="tab:red", marker="x", s=80, label="Abstenção proposta", zorder=5)
    axes[0].set(title="Auditoria temporal e abstenções propostas", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[0].grid(alpha=.25)
    axes[0].legend()

    axes[1].scatter(audit["width_relative_iqr"], audit["absolute_error_m"], c=audit["reliable_frames"], cmap="viridis", s=45)
    axes[1].axvline(BOX_AUDIT_MAX_RELATIVE_IQR, color="tab:red", linestyle="--", label="Limite IQR relativo")
    axes[1].set(title="Dispersão da largura versus erro", xlabel="IQR da largura / mediana", ylabel="Erro absoluto (m)")
    axes[1].grid(alpha=.25)
    axes[1].legend()

    axes[2].bar(quality_comparison["variant"], quality_comparison["mae_m"], label="MAE")
    axes[2].scatter(quality_comparison["variant"], quality_comparison["rmse_m"], color="tab:red", label="RMSE", zorder=3)
    for index, row in quality_comparison.reset_index(drop=True).iterrows():
        axes[2].text(index, row["mae_m"]+.15, f"cobertura={row['coverage_all_labeled_seconds_pct']:.1f}%", ha="center", fontsize=9)
    axes[2].set(title="Erro e cobertura antes/depois do controle de qualidade", ylabel="Erro (m)")
    axes[2].tick_params(axis="x", rotation=15)
    axes[2].grid(axis="y", alpha=.25)
    axes[2].legend()
    fig.savefig(AUDIT_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    print(f"Equação provisória: distância_m = {constant:.8g} × largura_px ^ ({exponent:.6f})")
    print(f"Segundos rejeitados pela regra operacional: {int(audit['quality_abstained'].sum())}/{len(audit)}")
    print("Resultado científico primário continua sendo a versão OOF sem filtro.")
    display(quality_comparison.round(3))
    print("Maiores erros para auditoria:")
    display(top_errors[[
        "second", "video_time_s", "reference_distance_nominal_m", "oof_estimate_m",
        "absolute_error_m", "reliable_frames", "width_median_px",
        "width_relative_iqr", "temporal_hampel_score", "quality_reason",
    ]].round(3))
    print("Métricas por bloco:")
    display(block_metrics.round(3))
    print("Arquivos salvos em:", OUTPUT_DIR)
else:
    print("Auditoria de robustez do modelo de largura bloqueada.")


## 8.8. Consolidação final dos métodos

Execute após a seção 8.7, alterando `RUN_FINAL_REPORT = True`. Esta etapa não executa inferência. Ela reúne os resultados do Depth Pro e do estimador por largura, preservando a distinção entre métricas descritivas e validação bloqueada fora da amostra.

O relatório seleciona como resultado científico principal o `box_width_log` OOF sem filtro. A versão com abstenção é apresentada separadamente como variante operacional. A equação ajustada em todos os segundos válidos é salva para demonstração e uso futuro, mas não substitui as métricas OOF.


In [ ]:
if RUN_FINAL_REPORT:
    required_files = {
        "depth_summary": OUTPUT_DIR / "method_comparison_summary.csv",
        "depth_second": OUTPUT_DIR / "method_comparison_per_second.csv",
        "box_predictions": OUTPUT_DIR / "box_distance_blocked_cv_predictions.csv",
        "quality": OUTPUT_DIR / "box_width_quality_comparison.csv",
        "audit": OUTPUT_DIR / "box_width_robustness_audit.csv",
        "model": OUTPUT_DIR / "box_width_model_provisional.json",
    }
    missing = [str(file) for file in required_files.values() if not file.exists()]
    if missing:
        raise FileNotFoundError("Arquivos ausentes; execute as seções 8.1, 8.6 e 8.7:\n" + "\n".join(missing))

    depth_summary = pd.read_csv(required_files["depth_summary"])
    depth_second = pd.read_csv(required_files["depth_second"])
    box_predictions = pd.read_csv(required_files["box_predictions"])
    quality_summary = pd.read_csv(required_files["quality"])
    robustness_audit = pd.read_csv(required_files["audit"])
    with required_files["model"].open("r", encoding="utf-8") as handle:
        selected_model = json.load(handle)
    with METRICS_JSON.open("r", encoding="utf-8") as handle:
        temporal_metrics_saved = json.load(handle)

    final_rows = []
    labeled_seconds = max(1, int(math.ceil(float(temporal_metrics_saved["telemetry_labeled_frames"]) / NATIVE_FPS)))
    depth_second_coverage_pct = 100.0 * depth_second["second"].nunique() / labeled_seconds
    depth_per_second = depth_summary.loc[depth_summary["level"] == "per_second_primary"]
    for _, row in depth_per_second.iterrows():
        final_rows.append({
            "method": row["method"],
            "role": "baseline_comparator",
            "evaluation": "full_sequence_per_second_descriptive",
            "coverage_pct": float(depth_second_coverage_pct),
            "n_seconds": int(row["n"]),
            "mae_m": float(row["mae_m"]),
            "rmse_m": float(row["rmse_m"]),
            "median_ae_m": float(row["median_ae_m"]),
            "mape_pct": float(row["mape_pct"]),
            "bias_m": float(row["bias_m"]),
            "r2": float(row["r2"]),
            "spearman": float(row["spearman"]),
            "within_1m_pct": float(row["within_1m_pct_emitted"]),
            "within_20pct_pct": float(row["within_20pct_pct_emitted"]),
            "legacy_5m_or_20pct_pct": float(row["legacy_5m_or_20pct_pct_emitted"]),
        })
    for _, row in quality_summary.iterrows():
        is_gated = row["variant"] == "box_width_log_oof_quality_gated"
        final_rows.append({
            "method": "Largura YOLO — OOF com abstenção" if is_gated else "Largura YOLO — OOF primário",
            "role": "operational_secondary" if is_gated else "selected_primary",
            "evaluation": "leave_one_20s_block_out",
            "coverage_pct": float(row["coverage_all_labeled_seconds_pct"]),
            "n_seconds": int(row["n"]),
            "mae_m": float(row["mae_m"]),
            "rmse_m": float(row["rmse_m"]),
            "median_ae_m": float(row["median_ae_m"]),
            "mape_pct": float(row["mape_pct"]),
            "bias_m": float(row["bias_m"]),
            "r2": float(row["r2"]),
            "spearman": float(row["spearman"]),
            "within_1m_pct": float(row["within_1m_pct_emitted"]),
            "within_20pct_pct": float(row["within_20pct_pct_emitted"]),
            "legacy_5m_or_20pct_pct": float(row["legacy_5m_or_20pct_pct_emitted"]),
        })
    final_comparison = pd.DataFrame(final_rows)
    FINAL_COMPARISON_CSV = OUTPUT_DIR / "final_method_comparison.csv"
    FINAL_PLOT = OUTPUT_DIR / "final_method_comparison.png"
    atomic_csv(final_comparison, FINAL_COMPARISON_CSV)

    primary_box = final_comparison.loc[final_comparison["role"] == "selected_primary"].iloc[0]
    gated_box = final_comparison.loc[final_comparison["role"] == "operational_secondary"].iloc[0]
    summary_payload = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "scientific_primary": primary_box.to_dict(),
        "operational_secondary": gated_box.to_dict(),
        "provisional_equation": selected_model,
        "interpretation": {
            "depth_pro": "Evaluated comparator; insufficient metric performance in this sequence.",
            "box_width": "Selected estimator under internal temporally blocked validation.",
            "external_validation": "Not available; coefficients remain provisional.",
        },
        "limitations": selected_model.get("limitations", []),
    }
    with FINAL_SUMMARY_JSON.open("w", encoding="utf-8") as handle:
        json.dump(summary_payload, handle, indent=2, ensure_ascii=False, allow_nan=True)

    primary_predictions = box_predictions.loc[
        box_predictions["model"] == "box_width_log"
    ].sort_values("video_time_s")
    fig, axes = plt.subplots(3, 1, figsize=(14, 16), constrained_layout=True)
    axes[0].plot(depth_second["video_time_s"], depth_second["reference_distance_nominal_m"], color="black", linewidth=2.2, label="Referência")
    axes[0].plot(depth_second["video_time_s"], depth_second["estimated_distance_m"], color="tab:orange", alpha=.8, label="Depth calibrado")
    axes[0].plot(primary_predictions["video_time_s"], primary_predictions["oof_estimate_m"], color="tab:blue", label="Largura YOLO — OOF")
    abstained_flag = robustness_audit["quality_abstained"]
    if abstained_flag.dtype != bool:
        abstained_flag = abstained_flag.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    rejected = robustness_audit.loc[abstained_flag]
    axes[0].scatter(rejected["video_time_s"], rejected["oof_estimate_m"], color="tab:red", marker="x", s=75, label="Abstenção proposta")
    axes[0].set(title="Comparação temporal final", xlabel="Tempo no vídeo (s)", ylabel="Distância (m)")
    axes[0].grid(alpha=.25)
    axes[0].legend()

    axes[1].bar(final_comparison["method"], final_comparison["mae_m"], label="MAE")
    axes[1].scatter(final_comparison["method"], final_comparison["rmse_m"], color="tab:red", label="RMSE", zorder=3)
    axes[1].set(title="Erro por método", ylabel="Erro (m)")
    axes[1].tick_params(axis="x", rotation=20)
    axes[1].grid(axis="y", alpha=.25)
    axes[1].legend()

    axes[2].scatter(final_comparison["coverage_pct"], final_comparison["mae_m"], s=90)
    for _, row in final_comparison.iterrows():
        axes[2].annotate(row["method"], (row["coverage_pct"], row["mae_m"]), xytext=(5, 5), textcoords="offset points", fontsize=9)
    axes[2].set(title="Cobertura versus MAE", xlabel="Cobertura (%)", ylabel="MAE (m)")
    axes[2].grid(alpha=.25)
    fig.savefig(FINAL_PLOT, dpi=180, bbox_inches="tight")
    plt.show()

    equation = selected_model["equation"]
    constant = selected_model["constant"]
    exponent = selected_model["exponent"]
    report_text = f"""# Relatório final — estimação de distância do drone

## Resultado selecionado

- Método científico principal: largura da bounding box YOLO com validação OOF por blocos de 20 segundos.
- Equação provisória de implantação: `distância_m = {constant:.8g} × largura_px ^ ({exponent:.6f})`.
- MAE OOF primário: `{primary_box['mae_m']:.3f} m`.
- RMSE OOF primário: `{primary_box['rmse_m']:.3f} m`.
- R² OOF primário: `{primary_box['r2']:.3f}`.
- Cobertura primária: `{primary_box['coverage_pct']:.3f}%`.

## Variante operacional

- MAE com controle de qualidade: `{gated_box['mae_m']:.3f} m`.
- RMSE com controle de qualidade: `{gated_box['rmse_m']:.3f} m`.
- R² com controle de qualidade: `{gated_box['r2']:.3f}`.
- Cobertura com controle de qualidade: `{gated_box['coverage_pct']:.3f}%`.

## Interpretação

O Depth Pro foi preservado como comparador experimental, mas não manteve desempenho métrico suficiente. A largura da caixa YOLO apresentou o melhor resultado interno. A versão sem filtro constitui o resultado científico principal; a versão com abstenção é uma análise operacional secundária.

## Limitações

- Único voo e ausência de validação externa.
- Necessidade de zoom fixo e resolução de 3840 × 2160.
- Dependência da aparência e orientação do mesmo modelo de drone.
- Apenas dois segundos válidos acima de 40 m.
- Quadros adjacentes e telemetria interpolada não são observações independentes.
- A equação ajustada em todos os dados serve para implantação provisória; seu desempenho é estimado pelas predições OOF.
"""
    FINAL_REPORT_MD.write_text(report_text, encoding="utf-8")

    print("Comparação consolidada:")
    display(final_comparison.round(3))
    print("Relatório:", FINAL_REPORT_MD)
    print("Resumo JSON:", FINAL_SUMMARY_JSON)
else:
    print("Consolidação final bloqueada.")


## 9. Vídeo final com o estimador selecionado

Depois da seção 8.7, altere `RUN_FINAL_ANNOTATED_VIDEO = True` e execute esta seção. O vídeo utiliza a equação ajustada em todos os segundos válidos e aplica a regra de abstenção. Ele é um produto demonstrativo; as métricas científicas permanecem sendo as predições OOF.


In [ ]:
def draw_final_box_panel(frame, frame_row, second_row, model_specification):
    box = parse_box(frame_row)
    has_second = second_row is not None
    abstained_value = True if not has_second else second_row.get("quality_abstained", True)
    if isinstance(abstained_value, str):
        abstained_value = abstained_value.strip().lower() in {"true", "1", "yes"}
    abstained = (not has_second) or bool(abstained_value)
    color = (0, 165, 255) if abstained else (0, 220, 0)
    if box is not None:
        x1, y1, x2, y2 = map(lambda value: int(round(value)), box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 4)
        confidence = float(frame_row.get("confidence", np.nan))
        label = f"drone conf={confidence:.3f}" if np.isfinite(confidence) else "drone"
        cv2.putText(frame, label, (x1, max(40, y1-12)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)

    reference = float(frame_row.get("reference_distance_nominal_m", np.nan))
    lines_panel = [
        f"video={float(frame_row.video_time_s):.3f}s | telemetria={float(frame_row.telemetry_time_s):.3f}s",
        f"referencia={reference:.2f} m" if np.isfinite(reference) else "referencia indisponivel",
    ]
    if not has_second:
        lines_panel.append("ABSTENCAO: segundo sem agregado YOLO confiavel")
    elif abstained:
        reason = str(second_row.get("quality_reason", "quality_gate")) or "quality_gate"
        lines_panel.append(f"ABSTENCAO: {reason}")
        lines_panel.append(
            f"largura mediana={float(second_row.width_median_px):.1f}px | caixas={int(second_row.reliable_frames)}"
        )
    else:
        estimate = float(second_row.final_estimate_m)
        error = abs(estimate-reference) if np.isfinite(reference) else np.nan
        lines_panel.append(
            f"estimativa largura YOLO={estimate:.2f} m | erro descritivo={error:.2f} m"
            if np.isfinite(error) else f"estimativa largura YOLO={estimate:.2f} m"
        )
        lines_panel.append(
            f"largura mediana={float(second_row.width_median_px):.1f}px | caixas={int(second_row.reliable_frames)}"
        )
    lines_panel.append(
        f"modelo: d={model_specification['constant']:.2f} * largura^{model_specification['exponent']:.4f}"
    )
    overlay = frame.copy()
    panel_h = 28 + 40*len(lines_panel)
    cv2.rectangle(overlay, (0, 0), (frame.shape[1], panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, .68, frame, .32, 0, frame)
    for index, line in enumerate(lines_panel):
        cv2.putText(frame, line, (22, 42+index*40), cv2.FONT_HERSHEY_SIMPLEX, 1.05, (255, 255, 255), 3, cv2.LINE_AA)
    return frame


if RUN_FINAL_ANNOTATED_VIDEO:
    audit_path = OUTPUT_DIR / "box_width_robustness_audit.csv"
    model_path = OUTPUT_DIR / "box_width_model_provisional.json"
    if not DEPTH_CSV.exists() or not audit_path.exists() or not model_path.exists():
        raise FileNotFoundError("Execute as seções 5–8.7 antes do vídeo final.")
    frame_results = pd.read_csv(DEPTH_CSV).set_index("frame_idx")
    second_results = pd.read_csv(audit_path).set_index("second")
    with model_path.open("r", encoding="utf-8") as handle:
        final_model = json.load(handle)
    second_results["final_estimate_m"] = (
        float(final_model["constant"])
        * second_results["width_median_px"].astype(float)**float(final_model["exponent"])
    )
    if len(frame_results) != TARGET_FRAMES:
        raise RuntimeError(f"Resultados incompletos: {len(frame_results)}/{TARGET_FRAMES}")

    output_width = WIDTH//2
    output_height = HEIGHT//2
    output_width -= output_width % 2
    output_height -= output_height % 2
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    writer = cv2.VideoWriter(
        str(FINAL_ANNOTATED_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"),
        NATIVE_FPS, (output_width, output_height),
    )
    if not writer.isOpened():
        raise RuntimeError("Não foi possível criar o vídeo final.")
    for frame_idx in range(END_FRAME_EXCLUSIVE):
        ok, frame = cap.read()
        if not ok:
            break
        row = frame_results.loc[frame_idx]
        second = int(math.floor(float(row.video_time_s)))
        second_row = second_results.loc[second] if second in second_results.index else None
        annotated = draw_final_box_panel(frame, row, second_row, final_model)
        annotated = cv2.resize(annotated, (output_width, output_height), interpolation=cv2.INTER_AREA)
        writer.write(annotated)
        rendered = frame_idx+1
        if rendered % max(1, int(NATIVE_FPS*10)) == 0:
            print(f"Render final: {rendered}/{TARGET_FRAMES}")
    cap.release()
    writer.release()
    print("Vídeo final arquivado em:", FINAL_ANNOTATED_VIDEO)
else:
    print("Vídeo final bloqueado.")


## 10. Registro final do protocolo

Execute ao final de cada modo. O JSON registra parâmetros, sincronização, geometria, limitações e caminhos dos produtos.


In [ ]:
protocol = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "experiment_role": "functional high-frequency temporal case study",
    "does_not_replace_primary_held_out_test": True,
    "mode": RUN_MODE,
    "video": str(VIDEO_PATH),
    "telemetry_file": str(TELEMETRY_PATH),
    "native_fps": NATIVE_FPS,
    "target_frames": TARGET_FRAMES,
    "detector": DETECTOR_TAG,
    "yolo_weights": str(YOLO_WEIGHTS),
    "yolo_imgsz": YOLO_IMGSZ,
    "depth_protocol": {
        "source": "full_frame",
        "roi_candidate": "s00_p25_m",
        "fixed_focal_px": FIXED_FOCAL_PX,
        "calibration": "log_linear",
        "role": "baseline_comparator",
        "intercept": CAL_INTERCEPT,
        "slope": CAL_SLOPE,
    },
    "synchronization": {
        "timecode_format": "HH:MM:SS:FF",
        "timecode_fps": TIMECODE_FPS,
        "takeoff_timecode": ORIGINAL_VIDEO_TAKEOFF_TIMECODE,
        "crop_start_timecode": CROP_START_TIMECODE,
        "original_video_takeoff_s": ORIGINAL_VIDEO_TAKEOFF_S,
        "telemetry_takeoff_s": TELEMETRY_TAKEOFF_S,
        "crop_start_in_original_video_s": CROP_START_IN_ORIGINAL_VIDEO_S,
        "telemetry_at_crop_start_s": TELEMETRY_AT_CROP_START_S,
        "frame_timestamp": "OpenCV CAP_PROP_POS_MSEC with frame-index fallback",
        "telemetry_interpolation": "linear",
    },
    "camera_geometry": {
        "camera_height_m": CAMERA_HEIGHT_M,
        "camera_to_home_horizontal_m": CAMERA_TO_HOME_HORIZONTAL_M,
        "unknown_azimuth_handling": "nominal orthogonal composition plus partial min/max bounds",
    },
    "abstention": {
        "confidence_min": YOLO_RELIABLE_CONF,
        "box_width_min_px": MIN_BOX_WIDTH_PX,
        "box_height_min_px": MIN_BOX_HEIGHT_PX,
        "valid_depth_pixels_min": MIN_VALID_DEPTH_PIXELS,
        "estimate_range_m": [MIN_ESTIMATE_M, MAX_ESTIMATE_M],
    },
    "independence_warning": "Adjacent frames and interpolated telemetry labels are not independent observations.",
    "outputs": {
        "yolo_csv": str(YOLO_CSV),
        "depth_csv": str(DEPTH_CSV),
        "metrics_json": str(METRICS_JSON),
        "per_second_csv": str(PER_SECOND_CSV),
        "zone_csv": str(ZONE_CSV),
        "legacy_depth_annotated_video": str(ANNOTATED_VIDEO),
        "final_box_width_annotated_video": str(FINAL_ANNOTATED_VIDEO),
        "final_report_markdown": str(FINAL_REPORT_MD),
        "final_summary_json": str(FINAL_SUMMARY_JSON),
    },
}
selected_model_path = OUTPUT_DIR / "box_width_model_provisional.json"
if selected_model_path.exists():
    with selected_model_path.open("r", encoding="utf-8") as handle:
        protocol["selected_distance_estimator"] = json.load(handle)
else:
    protocol["selected_distance_estimator"] = {
        "status": "not_generated",
        "expected_file": str(selected_model_path),
    }
protocol["selected_method_evaluation"] = {
    "scientific_primary": "box_width_log OOF leave-one-20s-block-out without quality filtering",
    "operational_secondary": "same OOF predictions with fixed quality-abstention rule",
    "deployment_equation_metrics_warning": "Equation fitted on all valid seconds; performance must be reported from OOF predictions.",
    "zoom_requirement": "fixed zoom; zoomed original tail excluded from primary evaluation",
}
with PROTOCOL_JSON.open("w", encoding="utf-8") as handle:
    json.dump(protocol, handle, indent=2, ensure_ascii=False)
print(json.dumps(protocol, indent=2, ensure_ascii=False))


## Ordem de execução resumida

### Piloto corrigido já processado

1. Reinicie o kernel e execute as seções **1–4**.
2. Execute a seção **4.1** para reaproveitar YOLO e Depth Pro e recalcular apenas as referências.
3. Mantenha YOLO e Depth como `False`; execute as seções **5–7** para carregar os CSVs corrigidos.
4. Ative `RUN_TEMPORAL_METRICS` e execute as seções **8–8.2**.
5. Para o diagnóstico amostral, ative também `RUN_ROI_DIAGNOSTIC` e execute a seção **8.3**.
6. Para comparar recortes, altere `RUN_CROP_DEPTH_DIAGNOSTIC = True` na célula complementar da seção 1 e execute a seção **8.4**.
7. Para testar calibração fora da amostra sem nova inferência, altere `RUN_CROP_CALIBRATION_CV = True` e execute a seção **8.5**.
8. Para validar a caixa YOLO em toda a sequência, altere `RUN_FULL_BOX_CALIBRATION_CV = True` e execute a seção **8.6**.
9. Para auditar o modelo de largura e extrair a equação, altere `RUN_BOX_ROBUSTNESS_AUDIT = True` e execute a seção **8.7**.
10. Para gerar a consolidação, altere `RUN_FINAL_REPORT = True` e execute a seção **8.8**.
11. Para o vídeo demonstrativo final, altere `RUN_FINAL_ANNOTATED_VIDEO = True` e execute as seções **9–10**.

### Execução integral

Altere `RUN_MODE = "full"`. Reaproveite o YOLO e o Depth já concluídos mantendo os dois controles de inferência como `False`. As seções 8–8.2 não fazem inferência. A seção 8.3 processa uma amostra estratificada de até 60 quadros; a seção 8.4 reutiliza a mesma amostra em recortes de 512 e 1024 px. Ambas permitem retomada. A seção 8.5 apenas recalibra os CSVs existentes com validação por blocos temporais. A seção 8.6 usa toda a sequência, agrega por segundo e retira blocos de 20 s, sem executar nova inferência. A seção 8.7 audita erros e cobertura, testa uma regra fixa de abstenção e salva a equação provisória. A seção 8.8 consolida os métodos e gera o relatório final. A seção 9 renderiza o vídeo do método selecionado.

Os resultados corrigidos ficam na subpasta `sync_tc_00000225_telemetry_004800`. Os CSVs antigos permanecem preservados na pasta anterior. Não use `FORCE_RESTART = True` salvo quando desejar recalcular as inferências da pasta corrigida.